In [1]:
import os
import re
import json
import math
import html
import warnings
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

# =========================
# CẤU HÌNH CHUNG
# =========================
NOTEBOOK_VERSION = "preprocessing_v2"
RUN_EMBEDDING = False            # bật khi bạn thật sự muốn sinh embedding
RUN_SECTION_EMBEDDING = False    # bật khi muốn embed từng chunk cho chatbot/RAG
SAVE_INTERMEDIATE = True

PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "data_topcv"

RAW_INPUT_CANDIDATES = [
    Path("/mnt/data/topcv_all_fields_merged_2026-03-16_20-57-23.xlsx"),
    DATA_DIR / "topcv_all_fields_merged_2026-03-16_20-57-23.xlsx",
    PROJECT_DIR / "topcv_all_fields_merged_2026-03-16_20-57-23.xlsx",
]

RAW_INPUT_PATH = next((p for p in RAW_INPUT_CANDIDATES if p.exists()), None)
if RAW_INPUT_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy file raw input. Hãy cập nhật RAW_INPUT_CANDIDATES hoặc upload file đúng đường dẫn."
    )

OUTPUT_DIR = PROJECT_DIR / "outputs_preprocessing_v2"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
REPORT_DIR = OUTPUT_DIR / "reports"

for p in [OUTPUT_DIR, ARTIFACT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("PROJECT_DIR     :", PROJECT_DIR)
print("RAW_INPUT_PATH  :", RAW_INPUT_PATH)
print("OUTPUT_DIR      :", OUTPUT_DIR)
print("RUN_EMBEDDING   :", RUN_EMBEDDING)
print("RUN_SECTION_EMB :", RUN_SECTION_EMBEDDING)

NOTEBOOK_VERSION: preprocessing_v2
PROJECT_DIR     : D:\TTTN\Project
RAW_INPUT_PATH  : D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-16_20-57-23.xlsx
OUTPUT_DIR      : D:\TTTN\Project\outputs_preprocessing_v2
RUN_EMBEDDING   : False
RUN_SECTION_EMB : False


In [2]:
# Tuỳ chọn hiển thị / debug
# pd.set_option("display.max_rows", 100)
# pd.set_option("display.max_columns", 200)
# pd.set_option("display.width", 200)
# pd.set_option("display.max_colwidth", 300)

# 1. Load dữ liệu và tiền xử lý cơ bản

In [3]:
# =========================
# HELPER FUNCTIONS
# =========================

EMPTY_TOKENS = {
    "", "nan", "none", "null", "n/a", "na", "-", "--",
    "không rõ", "chưa cập nhật", "đang cập nhật", "not specified"
}
# Chuẩn hóa giá trị rỗng thành none
def normalize_empty_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if val_str.lower() in EMPTY_TOKENS:
        return None
    return val_str

# Lấy cột nếu tồn tại, nếu không trả về Series mặc định
def get_series(df: pd.DataFrame, col: str, default=None) -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)

# Lấy giá trị đầu tiên không rỗng trong danh sách
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode về dạng NFC
def normalize_unicode(text: str) -> str:
    if text is None:
        return ""
    return unicodedata.normalize("NFC", str(text))

# Loại bỏ html thay bằng khoảng trắng
def remove_html(text: str) -> str:
    text = normalize_unicode(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&nbsp;?", " ", text)
    return text

# Chuẩn hóa các ký hiệu đầu dòng, gạch ngang khác nhau về dạng "-"
def normalize_dash(text: str) -> str:
    return (
        text.replace("•", " - ")
            .replace("▪", " - ")
            .replace("✅", " - ")
            .replace("✔", " - ")
            .replace("★", " - ")
            .replace("●", " - ")
            .replace("−", "-")
            .replace("–", "-")
            .replace("—", "-")
    )

# Chuyển giá trị về string, nếu là None hoặc NaN thì trả về chuỗi rỗng
def safe_str(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()

# Loại bỏ phần tử trùng lặp trong list, giữ nguyên thứ tự xuất hiện đầu tiên
def deduplicate_list(values):
    out, seen = [], set()
    for v in values:
        if not v:
            continue
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

# Loại bỏ các dòng gần giống nhau trong văn bản nhiều dòng
def deduplicate_text_lines(text: str, min_key_len: int = 12) -> str:
    text = safe_str(text)
    if not text:
        return ""

    normalized = text.replace("\\n", "\n")
    raw_lines = [ln.strip() for ln in re.split(r"\n+", normalized) if ln.strip()]
    if not raw_lines:
        return text.strip()

    cleaned_lines = []
    seen = set()

    for ln in raw_lines:
        key = re.sub(r"^[\-\*\•\▪\+\d\.\)\( ]+", "", clean_text_strict(ln)) if "clean_text_strict" in globals() else ln.lower()
        if len(key) >= min_key_len and key in seen:
            continue
        if len(key) >= min_key_len:
            seen.add(key)
        cleaned_lines.append(ln)

    return "\n".join(cleaned_lines).strip()

# Làm sạch nhẹ – giữ cấu trúc, loại bỏ ký tự thừa, HTML, khoảng trắng dư thừa
def clean_text_light(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)

    text = text.replace("\\n", "\n")
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%]", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()

# Làm sạch nhưng giữ cấu trúc (đoạn văn, xuống dòng) nhiều hơn
def clean_text_preserve_structure(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)

    text = text.replace("\\n", "\n")
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = deduplicate_text_lines(text)

    return text.strip()

# Làm sạch tối đa – biến thành chuỗi liền mạch, lowercase, chỉ giữ ký tự cơ bản
def clean_text_strict(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text).lower()
    text = remove_html(text)
    text = normalize_dash(text)

    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[^\w\sÀ-ỹ\./\-\+\#]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Lưu DataFrame, ưu tiên Parquet, fallback CSV nếu có lỗi
def save_table(df: pd.DataFrame, base_path: Path) -> Path:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = base_path.with_suffix(".parquet")
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as e:
        csv_path = base_path.with_suffix(".csv")
        warnings.warn(
            f"Không lưu được parquet tại {parquet_path.name} ({type(e).__name__}: {e}). "
            f"Đã fallback sang CSV: {csv_path.name}"
        )
        df.to_csv(csv_path, index=False)
        return csv_path

In [4]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    ext = input_path.suffix.lower()
    if ext == ".xlsx":
        df = pd.read_excel(input_path)
    elif ext == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Định dạng file không hỗ trợ: {ext}")

    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df

raw_df = load_raw_data(RAW_INPUT_PATH)
display(raw_df.head(3))
print("\nColumns:\n", raw_df.columns.tolist())

[INFO] Loaded raw data: 325 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,detail_salary,address_list,detail_location,exp_list,detail_experience,deadline,tags,job_level,education_level,job_quantity,employment_type,working_addresses,working_times,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,12 - 20 triệu,12 - 20 triệu,Hồ Chí Minh (mới),Hồ Chí Minh,2 năm,2 năm,27/03/2026,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,20 - 30 triệu,Hà Nội,NaN,2 năm,2 năm,Còn 17 ngày để ứng tuyển,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập


Columns:
 ['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [5]:
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["job_url"] = get_series(df, "job_url")
    out["source_field_name"] = get_series(df, "source_field_name")
    out["field_count"] = get_series(df, "field_count")

    out["job_title_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "detail_title"), get_series(df, "title"))
    ]

    out["salary_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "detail_salary"), get_series(df, "salary_list"))
    ]
    out["location_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "detail_location"), get_series(df, "address_list"))
    ]
    out["working_addresses_raw"] = get_series(df, "working_addresses")
    out["working_times_raw"] = get_series(df, "working_times")
    out["experience_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "detail_experience"), get_series(df, "exp_list"))
    ]
    out["tags_raw"] = get_series(df, "tags")

    out["job_level_raw"] = get_series(df, "job_level")
    out["education_level_raw"] = get_series(df, "education_level")
    out["employment_type_raw"] = get_series(df, "employment_type")
    out["job_quantity"] = get_series(df, "job_quantity")
    out["deadline_raw"] = get_series(df, "deadline")

    out["description_raw"] = get_series(df, "desc_mota")
    out["requirements_raw"] = get_series(df, "desc_yeucau")
    out["benefits_raw"] = get_series(df, "desc_quyenloi")

    out["company_name_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "company_name_full"), get_series(df, "company_name"))
    ]
    out["company_url"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "company_url_from_job"), get_series(df, "company_url"))
    ]
    out["company_website_raw"] = get_series(df, "company_website")
    out["company_scale_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "company_scale_from_job"), get_series(df, "company_scale"))
    ]
    out["company_field_raw"] = get_series(df, "company_field_from_job")
    out["company_address_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(get_series(df, "company_address_from_job"), get_series(df, "company_address"))
    ]
    out["company_description_raw"] = get_series(df, "company_description")

    print(f"[INFO] After merging: {out.shape[0]} rows x {out.shape[1]} cols")
    return out.reset_index(drop=True)

df = merge_semantic_columns(raw_df)
display(df.head(3))
df.info()

[INFO] After merging: 325 rows x 25 cols


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,job_level_raw,education_level_raw,employment_type_raw,job_quantity,deadline_raw,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,Toàn thời gian,1 người,27/03/2026,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,Toàn thời gian,1 người,Còn 17 ngày để ứng tuyển,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập năm 2018, Edupia là Edtech lớn nhất Việt Nam về Tiếng Anh Online cho trẻ em, đồng thời là Top 50 Edtech Nổi bật nhất Đông Nam Á năm 2022 & 2023. Qua nhiều năm phát triển, Edupia đến..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,"(đã được cập 

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  325 non-null    str  
 1   source_field_name        325 non-null    str  
 2   field_count              325 non-null    int64
 3   job_title_raw            325 non-null    str  
 4   salary_raw               325 non-null    str  
 5   location_raw             325 non-null    str  
 6   working_addresses_raw    325 non-null    str  
 7   working_times_raw        294 non-null    str  
 8   experience_raw           325 non-null    str  
 9   tags_raw                 325 non-null    str  
 10  job_level_raw            325 non-null    str  
 11  education_level_raw      325 non-null    str  
 12  employment_type_raw      325 non-null    str  
 13  job_quantity             325 non-null    str  
 14  deadline_raw             325 non-null    str  
 15  description_raw  

# 2. Audit dữ liệu và làm sạch

In [6]:
df_clean = df.copy()

print(f"[INFO] df_raw shape   : {df.shape}")
print(f"[INFO] df_clean shape : {df_clean.shape}")

[INFO] df_raw shape   : (325, 25)
[INFO] df_clean shape : (325, 25)


In [ ]:
# Phân loại ngôn ngữ dựa trên việc có chứa ký tự tiếng Việt và/hoặc ký tự ASCII cơ bản
def detect_language_type(text: str) -> str:
    text = safe_str(text)
    if not text:
        return "unknown"

    has_vietnamese = bool(re.search(r"[À-ỹđĐ]", text))
    has_ascii_words = bool(re.search(r"[A-Za-z]", text))

    if has_vietnamese and has_ascii_words:
        return "mixed"
    if has_vietnamese:
        return "vi"
    if has_ascii_words:
        return "en"
    return "unknown"

raw_text_cols = [
    "job_title_raw", "salary_raw", "location_raw", "working_addresses_raw", "working_times_raw",
    "experience_raw", "tags_raw", "job_level_raw", "education_level_raw", "employment_type_raw",
    "deadline_raw", "description_raw", "requirements_raw", "benefits_raw",
    "company_name_raw", "company_url", "company_website_raw", "company_scale_raw",
    "company_field_raw", "company_address_raw", "company_description_raw"
]

for col in raw_text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(normalize_empty_value)

df_clean["job_title_raw_norm_for_dup"] = df_clean["job_title_raw"].apply(clean_text_strict)
df_clean["company_name_raw_norm_for_dup"] = df_clean["company_name_raw"].apply(clean_text_strict)

df_clean["is_duplicate_by_url"] = df_clean["job_url"].duplicated(keep=False)
df_clean["is_duplicate_soft"] = (
    df_clean
    .assign(
        dup_key=(
            df_clean["company_name_raw_norm_for_dup"].fillna("") + " || " +
            df_clean["job_title_raw_norm_for_dup"].fillna("")
        )
    )["dup_key"]
    .duplicated(keep=False)
)

df_clean["language_type"] = (
    df_clean["job_title_raw"].fillna("") + " " +
    df_clean["requirements_raw"].fillna("")
).apply(detect_language_type)

df_clean["dense_encoder_route"] = np.where(
    df_clean["language_type"].eq("en"),
    "multilingual",
    "phobert"
)

df_clean["has_minimum_content"] = (
    df_clean["job_title_raw"].fillna("").str.len().ge(3) &
    (
        df_clean["description_raw"].fillna("").str.len().ge(30) |
        df_clean["requirements_raw"].fillna("").str.len().ge(30)
    )
)

audit_report = pd.DataFrame({
    "n_rows": [len(df_clean)],
    "dup_by_url_ratio": [round(df_clean["is_duplicate_by_url"].mean(), 4)],
    "dup_soft_ratio": [round(df_clean["is_duplicate_soft"].mean(), 4)],
    "has_minimum_content_ratio": [round(df_clean["has_minimum_content"].mean(), 4)],
    "vi_ratio": [round(df_clean["language_type"].eq("vi").mean(), 4)],
    "mixed_ratio": [round(df_clean["language_type"].eq("mixed").mean(), 4)],
    "en_ratio": [round(df_clean["language_type"].eq("en").mean(), 4)],
})

display(audit_report)

missing_report = (
    df_clean.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_ratio")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(missing_report.head(20))

,n_rows,dup_by_url_ratio,dup_soft_ratio,has_minimum_content_ratio,vi_ratio,mixed_ratio,en_ratio
0,325,0.0,0.0615,1.0,0.0,0.8369,0.1631


,column,missing_ratio
0,company_website_raw,0.169231
1,working_times_raw,0.095385
2,company_field_raw,0.080000
3,company_description_raw,0.027692
4,job_title_raw,0.000000
5,job_url,0.000000
6,field_count,0.000000
7,source_field_name,0.000000
8,experience_raw,0.000000
9,tags_raw,0.000000


In [8]:
print("[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.")
display(df_clean[raw_text_cols].head(3))

[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.


,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,job_level_raw,education_level_raw,employment_type_raw,deadline_raw,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,Toàn thời gian,27/03/2026,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,Data Governance Specialist,20 - 30 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,Toàn thời gian,Còn 17 ngày để ứng tuyển,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập năm 2018, Edupia là Edtech lớn nhất Việt Nam về Tiếng Anh Online cho trẻ em, đồng thời là Top 50 Edtech Nổi bật nhất Đông Nam Á năm 2022 & 2023. Qua nhiều năm phát triển, Edupia đến..."
2,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Chuyên môn IT Project Manager,Nhân viên,Đại Học trở lên,Toàn thời gia

In [ ]:
# Tạo các cột *_clean_light và *_clean_struct phujc vụ cho các mục đích khác nhau:
# - *_clean_light: làm sạch nhẹ, giữ cấu trúc gần như nguyên vẹn, phù hợp cho việc hiển thị hoặc các tác vụ không yêu cầu chuẩn hóa quá mức.
# - *_clean_struct: làm sạch nhưng giữ cấu trúc (đoạn văn, xuống dòng) nhiều hơn, phù hợp cho các tác vụ cần giữ nguyên định dạng như RAG hoặc chatbot.
light_clean_cols = [
    "job_title_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_description_raw",
    "working_times_raw",
    "working_addresses_raw",
    "company_address_raw",
    "company_name_raw",
    "tags_raw"
]

for col in light_clean_cols:
    if col in df_clean.columns:
        df_clean[col.replace("_raw", "_clean_light")] = df_clean[col].apply(clean_text_light)
        df_clean[col.replace("_raw", "_clean_struct")] = df_clean[col].apply(clean_text_preserve_structure)

print("[INFO] Đã tạo xong các cột *_clean_light và *_clean_struct")

[INFO] Đã tạo xong các cột *_clean_light và *_clean_struct


In [ ]:
# Tạo các cột *_clean_strict với mức độ làm sạch cao nhất, phù hợp cho việc phân tích, trích xuất thông tin, hoặc các tác vụ NLP yêu cầu chuẩn hóa mạnh.
# Các cột này sẽ được sử dụng cho việc trích xuất thông tin, phân tích tần suất từ khóa, hoặc các tác vụ NLP yêu cầu chuẩn hóa mạnh.
strict_clean_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "tags_raw",
    "company_field_raw",
    "working_times_raw",
    "working_addresses_raw",
    "requirements_raw",
    "description_raw"
]

for col in strict_clean_cols:
    if col in df_clean.columns:
        df_clean[col.replace("_raw", "_clean_strict")] = df_clean[col].apply(clean_text_strict)

print("[INFO] Đã tạo xong các cột *_clean_strict")

[INFO] Đã tạo xong các cột *_clean_strict


In [11]:
clean_cols_created = [
    c for c in df_clean.columns
    if c.endswith("_clean_light") or c.endswith("_clean_strict") or c.endswith("_clean_struct")
]

print(f"[INFO] Số cột clean đã tạo: {len(clean_cols_created)}")
print(clean_cols_created)

[INFO] Số cột clean đã tạo: 33
['job_title_clean_light', 'job_title_clean_struct', 'description_clean_light', 'description_clean_struct', 'requirements_clean_light', 'requirements_clean_struct', 'benefits_clean_light', 'benefits_clean_struct', 'company_description_clean_light', 'company_description_clean_struct', 'working_times_clean_light', 'working_times_clean_struct', 'working_addresses_clean_light', 'working_addresses_clean_struct', 'company_address_clean_light', 'company_address_clean_struct', 'company_name_clean_light', 'company_name_clean_struct', 'tags_clean_light', 'tags_clean_struct', 'job_title_clean_strict', 'salary_clean_strict', 'location_clean_strict', 'experience_clean_strict', 'education_level_clean_strict', 'employment_type_clean_strict', 'job_level_clean_strict', 'tags_clean_strict', 'company_field_clean_strict', 'working_times_clean_strict', 'working_addresses_clean_strict', 'requirements_clean_strict', 'description_clean_strict']


In [12]:
preview_cols = [
    "job_title_raw", "job_title_clean_light", "job_title_clean_strict",
    "requirements_raw", "requirements_clean_light", "requirements_clean_struct", "requirements_clean_strict",
    "description_raw", "description_clean_light", "description_clean_struct", "description_clean_strict"
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(3))

,job_title_raw,job_title_clean_light,job_title_clean_strict,requirements_raw,requirements_clean_light,requirements_clean_struct,requirements_clean_strict,description_raw,description_clean_light,description_clean_struct,description_clean_strict
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...",- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,- at least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - bachelor s degree in finance/accounting ...,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...","- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-quali...","- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",- overseeing all financial planning reporting analysis for the bee office. - track management reports focusing on budget/estimate/actuals to ensure those are delivered on time and with high-qualit...
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,"− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...",- đã tốt nghiệp đh trở lên ưu tiên các chuyên ngành về công nghệ thông tin/ hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực data governance/ da...,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...",- xây dựng triển khai và duy trì khung data governance cho toàn hệ thống dữ liệu của công ty. - xây dựng chính sách tiêu chuẩn quy định về quản lý dữ liệu data quality data profiling data security...
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,"Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...","Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, P

In [13]:
empty_after_clean_df = pd.DataFrame({
    "column": preview_cols,
    "empty_ratio": [
        df_clean[col].fillna("").astype(str).str.strip().eq("").mean()
        for col in preview_cols
    ]
}).sort_values("empty_ratio", ascending=False)

display(empty_after_clean_df)

,column,empty_ratio
0,job_title_raw,0.0
1,job_title_clean_light,0.0
2,job_title_clean_strict,0.0
3,requirements_raw,0.0
4,requirements_clean_light,0.0
5,requirements_clean_struct,0.0
6,requirements_clean_strict,0.0
7,description_raw,0.0
8,description_clean_light,0.0
9,description_clean_struct,0.0


In [14]:
length_check_cols = [
    "description_clean_light",
    "requirements_clean_light",
    "benefits_clean_light",
    "company_description_clean_light",
    "description_clean_struct",
    "requirements_clean_struct"
]
length_check_cols = [c for c in length_check_cols if c in df_clean.columns]

for col in length_check_cols:
    df_clean[f"{col}_len"] = df_clean[col].fillna("").str.len()

display(
    df_clean[[f"{c}_len" for c in length_check_cols]].describe().T
)

,count,mean,std,min,25%,50%,75%,max
description_clean_light_len,325.0,1018.560000,780.434020,149.0,535.0,821.0,1209.0,6034.0
requirements_clean_light_len,325.0,858.947692,622.290418,118.0,485.0,708.0,1084.0,5829.0
benefits_clean_light_len,325.0,728.889231,435.602067,90.0,388.0,614.0,1037.0,3461.0
company_description_clean_light_len,325.0,932.972308,721.431706,0.0,437.0,756.0,1284.0,3699.0
description_clean_struct_len,325.0,1020.698462,782.230560,149.0,535.0,821.0,1209.0,6034.0
requirements_clean_struct_len,325.0,860.209231,622.692461,118.0,490.0,708.0,1089.0,5829.0


In [15]:
if SAVE_INTERMEDIATE:
    clean_checkpoint_path = save_table(df_clean, ARTIFACT_DIR / "jobs_cleaning_checkpoint")
    print(f"[INFO] Saved checkpoint: {clean_checkpoint_path}")

[INFO] Saved checkpoint: D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_cleaning_checkpoint.parquet


# 3. Normalize metadata

In [ ]:
# Danh sách các pattern thường thấy ở cuối title để loại bỏ phần noise không cần thiết
TITLE_TRAILING_NOISE_PATTERNS = [
    r"\b(lương|thu nhập|up to|mức lương|deal lương|thỏa thuận|thoả thuận)\b.*$",
    r"\b(đi làm ngay|urgent|hot|gấp|nhận việc ngay)\b.*$",
    r"\b(còn \d+ ngày.*|hạn nộp.*)$",
    r"\b\d+\+?\s*(năm|year)s?\s*(kinh nghiệm|experience)\b.*$",
    r"\s*\|\s*.*$",
]
# Danh sách các từ khóa thường thấy trong ngoặc đơn/ngoặc vuông nhưng không phải là phần quan trọng của title
BRACKET_NOISE_KEYWORDS = {
    "hà nội", "ha noi", "hồ chí minh", "ho chi minh", "tp hcm", "hcm", "đà nẵng", "da nang",
    "remote", "hybrid", "onsite", "on-site", "urgent", "gấp", "hot", "lương", "mức lương", "deal"
}
# Bản đồ đồng nghĩa để chuẩn hóa title về một số dạng phổ biến, giúp việc phân loại dễ dàng hơn
TITLE_SYNONYM_MAP = {
    "chuyên viên phân tích dữ liệu": "data analyst",
    "nhân viên phân tích dữ liệu": "data analyst",
    "data analyst": "data analyst",
    "business analyst": "business analyst",
    "chuyên viên phân tích nghiệp vụ": "business analyst",
    "data engineer": "data engineer",
    "big data engineer": "data engineer",
    "data architect": "data architect",
    "data scientist": "data scientist",
    "machine learning engineer": "machine learning engineer",
    "ml engineer": "machine learning engineer",
    "ai engineer": "ai engineer",
    "ai researcher": "ai researcher",
    "ai scientist": "ai scientist",
    "data governance specialist": "data governance specialist",
    "project manager": "project manager",
    "product owner": "product owner",
    "backend developer": "backend developer",
    "backend engineer": "backend developer",
    "frontend developer": "frontend developer",
    "fullstack developer": "fullstack developer",
    "full-stack developer": "fullstack developer",
}
# Các quy tắc để suy luận job family dựa trên việc có chứa các từ khóa nhất định trong title đã được chuẩn hóa
JOB_FAMILY_RULES = {
    "data_analytics": [
        "data analyst", "business analyst", "bi analyst", "business intelligence",
        "analytics", "reporting", "dashboard", "power bi", "tableau"
    ],
    "data_engineering": [
        "data engineer", "etl", "elt", "data warehouse", "data platform",
        "big data", "lakehouse", "data architect"
    ],
    "data_science_ml": [
        "data scientist", "machine learning engineer", "ml engineer", "ai engineer",
        "ai researcher", "ai scientist", "computer vision", "nlp", "llm", "gen ai"
    ],
    "data_governance_quality": [
        "data governance", "data quality", "data lineage", "master data"
    ],
    "product_project_ba": [
        "business analyst", "project manager", "product owner", "scrum master", "project coordinator"
    ],
    "software_engineering": [
        "backend developer", "frontend developer", "fullstack developer",
        "software engineer", "devops", "qa", "tester"
    ],
}
# Các quy tắc để suy luận seniority dựa trên việc có chứa các từ khóa nhất định trong title hoặc description đã được chuẩn hóa
SENIORITY_RULES = [
    ("director", ["director", "giám đốc"]),
    ("manager", ["manager", "quản lý", "head", "trưởng phòng"]),
    ("lead", ["team lead", "trưởng nhóm", "leader", "lead", "principal", "staff"]),
    ("senior", ["senior", "sr", "cao cấp"]),
    ("middle", ["middle", "mid", "intermediate"]),
    ("junior", ["junior", "jr", "fresher", "entry level", "nhân viên", "chuyên viên"]),
    ("intern", ["intern", "internship", "thực tập", "thực tập sinh"]),
]
# Pattern để loại bỏ phần noise liên quan đến địa điểm thường thấy ở cuối title
CITY_NOISE_PATTERN = re.compile(
    r"\s*[-|–—,]\s*(hà nội|ha noi|hồ chí minh|ho chi minh|tp hcm|hcm|đà nẵng|da nang|remote|hybrid)\s*$",
    flags=re.IGNORECASE
)
# Hàm để loại bỏ phần noise trong ngoặc đơn/ngoặc vuông nếu chứa các từ khóa thường thấy nhưng không phải là phần quan trọng của title
def strip_bracket_noise(text: str) -> str:
    def _repl(match):
        content = clean_text_strict(match.group(1))
        if any(k in content for k in BRACKET_NOISE_KEYWORDS):
            return " "
        return f" {match.group(1).strip()} "
    text = re.sub(r"\((.*?)\)", _repl, text)
    text = re.sub(r"\[(.*?)\]", _repl, text)
    return text

def normalize_job_title(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = clean_text_light(text)
    text = strip_bracket_noise(text)

    for pattern in TITLE_TRAILING_NOISE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    text = re.sub(CITY_NOISE_PATTERN, "", text).strip()
    text = re.sub(r"\s+", " ", text).strip(" -|,")
    text_lower = text.lower()

    return TITLE_SYNONYM_MAP.get(text_lower, text_lower)
# Hàm để suy luận job family dựa trên việc có chứa các từ khóa nhất định trong title đã được chuẩn hóa
def infer_job_family(job_title_canonical: str) -> str:
    jt = safe_str(job_title_canonical).lower()
    if not jt:
        return "unknown"

    for family, patterns in JOB_FAMILY_RULES.items():
        if any(p in jt for p in patterns):
            return family
    return "other"
# Hàm để suy luận seniority dựa trên việc có chứa các từ khóa nhất định trong title hoặc description đã được chuẩn hóa
def infer_seniority_from_text(text: str) -> str:
    t = clean_text_strict(text)
    if not t:
        return "unknown"

    for label, patterns in SENIORITY_RULES:
        if any(re.search(rf"(?<!\w){re.escape(p)}(?!\w)", t) for p in patterns):
            return label
    return "unknown"

In [17]:
df_clean["job_title_display"] = df_clean["job_title_raw"].apply(clean_text_light)
df_clean["job_title_canonical"] = df_clean["job_title_raw"].apply(normalize_job_title)
df_clean["job_family"] = df_clean["job_title_canonical"].apply(infer_job_family)
df_clean["seniority_from_title"] = df_clean["job_title_raw"].apply(infer_seniority_from_text)

display(
    df_clean[
        ["job_title_raw", "job_title_display", "job_title_canonical", "job_family", "seniority_from_title"]
    ].head(10)
)

,job_title_raw,job_title_display,job_title_canonical,job_family,seniority_from_title
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,junior
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,data_governance_quality,unknown
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,product_project_ba,manager
3,AI Engineer,AI Engineer,ai engineer,data_science_ml,unknown
4,AI Engineer,AI Engineer,ai engineer,data_science_ml,unknown
5,Data Analyst,Data Analyst,data analyst,data_analytics,unknown
6,Data Engineer,Data Engineer,data engineer,data_engineering,unknown
7,Fresher Data Engineer,Fresher Data Engineer,fresher data engineer,data_engineering,junior
8,Data Analyst Teamleader (Collection Analytics),Data Analyst Teamleader (Collection Analytics),data analyst teamleader collection analytics,data_analytics,unknown
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,data analyst teamleader collection analytics,data_analytics,unknown


In [18]:
VIETNAM_CITIES = sorted({
    "hà nội", "ha noi", "hồ chí minh", "ho chi minh", "đà nẵng", "da nang",
    "hải phòng", "hai phong", "cần thơ", "can tho", "huế", "hue",
    "an giang", "bắc ninh", "bac ninh", "cà mau", "ca mau", "cao bằng", "cao bang",
    "đắk lắk", "dak lak", "điện biên", "dien bien", "đồng nai", "dong nai",
    "đồng tháp", "dong thap", "gia lai", "hà tĩnh", "ha tinh", "hưng yên", "hung yen",
    "khánh hòa", "khanh hoa", "lai châu", "lai chau", "lâm đồng", "lam dong",
    "lạng sơn", "lang son", "lào cai", "lao cai", "nghệ an", "nghe an",
    "ninh bình", "ninh binh", "phú thọ", "phu tho", "quảng ngãi", "quang ngai",
    "quảng ninh", "quang ninh", "quảng trị", "quang tri", "sơn la", "son la",
    "tây ninh", "tay ninh", "thái nguyên", "thai nguyen", "thanh hóa", "thanh hoa",
    "tuyên quang", "tuyen quang", "vĩnh long", "vinh long"
})

In [19]:
CITY_ALIAS_MAP = {
    "hà nội": "hà nội",
    "ha noi": "hà nội",
    "hn": "hà nội",
    "hồ chí minh": "hồ chí minh",
    "ho chi minh": "hồ chí minh",
    "tp hcm": "hồ chí minh",
    "hcm": "hồ chí minh",
    "hồ chí minh city": "hồ chí minh",
    "đà nẵng": "đà nẵng",
    "da nang": "đà nẵng",
    "hải phòng": "hải phòng",
    "hai phong": "hải phòng",
    "cần thơ": "cần thơ",
    "can tho": "cần thơ",
    "thừa thiên huế": "huế",
    "huế": "huế",
    "hue": "huế",
    
    "bắc ninh": "bắc ninh",
    "bac ninh": "bắc ninh",
    "đồng nai": "đồng nai",
    "dong nai": "đồng nai",
    "khánh hòa": "khánh hòa",
    "khanh hoa": "khánh hòa",
    "quảng ninh": "quảng ninh",
    "quang ninh": "quảng ninh",
    "cà mau": "cà mau",
    "ca mau": "cà mau",
    "cao bằng": "cao bằng",
    "cao bang": "cao bang",
    "đắk lắk": "đắk lắk",
    "dak lak": "đắk lắk",
    "điện biên": "điện biên",
    "dien bien": "điện biên",
    "đồng tháp": "đồng tháp",
    "dong thap": "đồng tháp",
    "gia lai": "gia lai",
    "hà tĩnh": "hà tĩnh",
    "ha tinh": "hà tĩnh",
    "hưng yên": "hưng yên",
    "hung yen": "hưng yên",
    "lai châu": "lai châu",
    "lai chau": "lai châu",
    "lâm đồng": "lâm đồng",
    "lam dong": "lâm đồng",
    "lạng sơn": "lạng sơn",
    "lang son": "lạng sơn",
    "lào cai": "lào cai",
    "lao cai": "lào cai",
    "nghệ an": "nghệ an",
    "nghe an": "nghệ an",
    "ninh bình": "ninh bình",
    "ninh binh": "ninh bình",
    "phú thọ": "phú thọ",
    "phu tho": "phú thọ",
    "quảng ngãi": "quảng ngãi",
    "quang ngai": "quảng ngãi",
    "quảng ninh": "quảng ninh",
    "quang ninh": "quảng ninh",
    "quảng trị": "quảng trị",
    "quang tri": "quảng trị",
    "sơn la": "sơn la",
    "son la": "sơn la",
    "tây ninh": "tây ninh",
    "tay ninh": "tây ninh",
    "thái nguyên": "thái nguyên",
    "thai nguyen": "thái nguyên",
    "thanh hóa": "thanh hóa",
    "thanh hoa": "thanh hóa",
    "tuyên quang": "tuyên quang",
    "tuyen quang": "tuyên quang",
    "vĩnh long": "vĩnh long",
    "vinh long": "vĩnh long"
}

In [ ]:
CITY_ALIASES_SORTED = sorted(CITY_ALIAS_MAP.keys(), key=len, reverse=True)

WORK_MODE_RULES = [
    ("remote", ["remote", "work from home", "wfh", "làm việc từ xa"]),
    ("hybrid", ["hybrid", "làm việc kết hợp", "làm việc linh hoạt"]),
    ("onsite", ["onsite", "on-site", "tại văn phòng", "làm việc tại văn phòng"]),
]

DISTRICT_PATTERN = re.compile(r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+')
# Hàm để phát hiện thành phố từ văn bản, ưu tiên tìm kiếm các alias đã được chuẩn hóa, tránh false positive từ các từ có chứa tên thành phố nhưng không phải là địa điểm (ví dụ: "hà nội" trong title job)
def detect_city_from_text(text: str):
    t = clean_text_strict(text)
    if not t:
        return None
    for alias in CITY_ALIASES_SORTED:
        if re.search(rf"(?<!\w){re.escape(alias)}(?!\w)", t):
            return CITY_ALIAS_MAP[alias]
    return None
# Hàm để suy luận work mode dựa trên việc có chứa các từ khóa liên quan đến remote/hybrid/onsite trong văn bản đã được chuẩn hóa
def infer_work_mode(*texts) -> str:
    merged = " ".join(clean_text_strict(t) for t in texts if normalize_empty_value(t))
    if not merged:
        return "unknown"
    for mode, patterns in WORK_MODE_RULES:
        if any(p in merged for p in patterns):
            return mode
    return "unknown"
# Hàm để kiểm tra xem văn bản có chứa thông tin về nhiều địa điểm hay không, dựa trên việc có chứa từ "và" kết hợp với "nơi khác" hoặc có chứa tên của nhiều thành phố khác nhau
def has_multi_location(text: str) -> int:
    t = clean_text_strict(text)
    if not t:
        return 0
    if "và" in t and "nơi khác" in t:
        return 1
    n_city_hits = sum(1 for alias in CITY_ALIASES_SORTED if alias in t)
    return int(n_city_hits >= 2)

In [ ]:
# Hàm để phân tích và trích xuất thông tin địa điểm làm việc từ văn bản thô, bao gồm việc phát hiện thành phố, quận/huyện, làm sạch văn bản và xác định xem có phải là multi-location hay không
def parse_working_address(raw_text: str) -> dict:
    text = clean_text_preserve_structure(raw_text)
    if not text:
        return {
            "location_city": None,
            "location_district": None,
            "working_address_clean": "",
            "is_multi_location": 0,
        }

    city = detect_city_from_text(text)
    district_match = DISTRICT_PATTERN.search(text)
    district = district_match.group(0).strip() if district_match else None

    text = re.sub(r"\s+", " ", text).strip(" -,")
    return {
        "location_city": city,
        "location_district": district,
        "working_address_clean": text,
        "is_multi_location": has_multi_location(text),
    }

def normalize_location(location_raw: str, working_addresses_raw: str) -> str:
    loc = clean_text_light(location_raw)
    addr_info = parse_working_address(working_addresses_raw)

    if loc:
        loc_city = detect_city_from_text(loc)
        if loc_city:
            return loc_city

    if addr_info["location_city"]:
        return addr_info["location_city"]

    return clean_text_strict(loc) if loc else ""

ABS_DATE_PATTERNS = [
    re.compile(r"(\d{1,2})[/-](\d{1,2})[/-](\d{4})"),
    re.compile(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})"),
]
# Hàm để phân tích và trích xuất thông tin deadline từ văn bản thô, bao gồm việc nhận diện ngày tuyệt đối, ngày tương đối (còn X ngày), và phân loại loại deadline cũng như tính toán số ngày còn lại đến deadline
def parse_deadline(raw: str, reference_date=None) -> dict:
    reference_date = reference_date or datetime.today().date()
    text = clean_text_light(raw)
    if not text:
        return {
            "deadline_clean": "",
            "deadline_date": None,
            "days_to_deadline": None,
            "deadline_type": "unknown",
            "is_expired": None,
        }

    text_strict = clean_text_strict(text)

    for pattern in ABS_DATE_PATTERNS:
        match = pattern.search(text)
        if match:
            try:
                if len(match.group(1)) == 4:
                    y, m, d = map(int, match.groups())
                else:
                    d, m, y = map(int, match.groups())
                dt = datetime(y, m, d).date()
                days_left = (dt - reference_date).days
                return {
                    "deadline_clean": text,
                    "deadline_date": dt.isoformat(),
                    "days_to_deadline": days_left,
                    "deadline_type": "absolute_date",
                    "is_expired": int(days_left < 0),
                }
            except Exception:
                pass

    rel_match = re.search(r"còn\s*(\d+)\s*ngày", text_strict)
    if rel_match:
        days_left = int(rel_match.group(1))
        dt = reference_date + timedelta(days=days_left)
        return {
            "deadline_clean": text,
            "deadline_date": dt.isoformat(),
            "days_to_deadline": days_left,
            "deadline_type": "relative_days",
            "is_expired": 0,
        }

    return {
        "deadline_clean": text,
        "deadline_date": None,
        "days_to_deadline": None,
        "deadline_type": "text_only",
        "is_expired": None,
    }

In [22]:
addr_parsed = df_clean["working_addresses_raw"].apply(parse_working_address)

df_clean["working_address_clean"] = addr_parsed.apply(lambda x: x["working_address_clean"])
df_clean["location_city"] = addr_parsed.apply(lambda x: x["location_city"])
df_clean["location_district"] = addr_parsed.apply(lambda x: x["location_district"])
df_clean["is_multi_location"] = addr_parsed.apply(lambda x: x["is_multi_location"])

df_clean["location_norm"] = df_clean.apply(
    lambda row: normalize_location(row.get("location_raw"), row.get("working_addresses_raw")),
    axis=1
)

df_clean["work_mode"] = df_clean.apply(
    lambda row: infer_work_mode(
        row.get("job_title_raw"),
        row.get("working_times_raw"),
        row.get("description_raw"),
        row.get("requirements_raw")
    ),
    axis=1
)

deadline_parsed = df_clean["deadline_raw"].apply(parse_deadline)
deadline_cols = ["deadline_clean", "deadline_date", "days_to_deadline", "deadline_type", "is_expired"]
for col in deadline_cols:
    df_clean[col] = deadline_parsed.apply(lambda x: x[col])

display(
    df_clean[
        [
            "location_raw", "working_addresses_raw", "location_city", "location_district",
            "location_norm", "is_multi_location", "work_mode",
            "deadline_raw", "deadline_date", "days_to_deadline", "deadline_type"
        ]
    ].head(10)
)

,location_raw,working_addresses_raw,location_city,location_district,location_norm,is_multi_location,work_mode,deadline_raw,deadline_date,days_to_deadline,deadline_type
0,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,27/03/2026,2026-03-27,7,absolute_date
1,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Còn 17 ngày để ứng tuyển,2026-04-06,17,relative_days
2,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Còn 24 ngày để ứng tuyển,2026-04-13,24,relative_days
3,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,30/04/2026,2026-04-30,41,absolute_date
4,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Landmark 72, Keangnam Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,11/05/2026,2026-05-11,52,absolute_date
5,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, E6 Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,08/04/2026,2026-04-08,19,absolute_date
6,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,09/04/2026,2026-04-09,20,absolute_date
7,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ) - Hà Nội: FPT Tower, số 10 ...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,09/04/2026,2026-04-09,20,absolute_date
8,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,10/04/2026,2026-04-10,21,absolute_date
9,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,10/04/2026,2026-04-10,21,absolute_date


In [23]:
display(df_clean["location_norm"].value_counts(dropna=False).head(20))
display(df_clean["work_mode"].value_counts(dropna=False))
display(df_clean["deadline_type"].value_counts(dropna=False))

location_norm
hà nội         231
hồ chí minh     76
đà nẵng          8
hưng yên         3
cần thơ          2
phú thọ          2
bắc ninh         1
vĩnh long        1
hải phòng        1
Name: count, dtype: int64

work_mode
unknown    298
remote      11
hybrid       9
onsite       7
Name: count, dtype: int64

deadline_type
absolute_date    312
relative_days     13
Name: count, dtype: int64

In [24]:
def clean_salary(raw: str) -> str:
    text = normalize_empty_value(raw)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = text.lower()
    text = text.replace("thoả", "thỏa")
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_numeric_token(num_text: str):
    token = safe_str(num_text)
    token = re.sub(r"[^\d,\.]", "", token)
    if not token:
        return None

    if token.count(",") and token.count("."):
        if token.rfind(",") > token.rfind("."):
            token = token.replace(".", "").replace(",", ".")
        else:
            token = token.replace(",", "")
    elif token.count(","):
        parts = token.split(",")
        if all(p.isdigit() for p in parts) and len(parts[-1]) == 3:
            token = "".join(parts)
        else:
            token = token.replace(",", ".")
    elif token.count(".") > 1:
        token = token.replace(".", "")

    try:
        return float(token)
    except Exception:
        return None

def detect_salary_multiplier(text: str, currency: str) -> float:
    if currency == "usd":
        return 1.0
    if any(k in text for k in ["triệu", "trieu", "tr "]):
        return 1_000_000
    if re.search(r"(?<!\w)tr(?!\w)", text):
        return 1_000_000
    if any(k in text for k in ["nghìn", "ngàn"]):
        return 1_000
    if re.search(r"(?<!\w)k(?!\w)", text):
        return 1_000
    return 1.0

def parse_salary_range(raw: str) -> dict:
    text = clean_salary(raw)
    if not text:
        return {
            "salary_clean": "",
            "salary_min_value": None,
            "salary_max_value": None,
            "salary_currency": "unknown",
            "salary_period": "unknown",
            "salary_type": "unknown",
            "salary_is_negotiable": 0,
            "salary_min_vnd_month": None,
            "salary_max_vnd_month": None,
        }

    currency = "usd" if ("usd" in text or "$" in text) else "vnd"

    period = "month"
    if re.search(r"(/|\b)mỗi\s*năm|\bnăm\b", text):
        period = "year"
    elif re.search(r"(/|\b)mỗi\s*ngày|\bngày\b", text):
        period = "day"
    elif re.search(r"(/|\b)mỗi\s*giờ|\bgiờ\b", text):
        period = "hour"

    negotiable = int(any(k in text for k in ["thỏa thuận", "thoả thuận", "deal lương", "negotiable"]))

    salary_type = "unknown"
    salary_min = None
    salary_max = None

    range_match = re.search(r"(\d[\d,\.]*)\s*-\s*(\d[\d,\.]*)", text)
    if range_match:
        salary_min = parse_numeric_token(range_match.group(1))
        salary_max = parse_numeric_token(range_match.group(2))
        salary_type = "range"
    else:
        up_to_match = re.search(r"(up to|tới|đến)\s*(\d[\d,\.]*)", text)
        above_match = re.search(r"(trên|from|từ|>=)\s*(\d[\d,\.]*)", text)
        single_match = re.search(r"(\d[\d,\.]*)", text)

        if up_to_match:
            salary_max = parse_numeric_token(up_to_match.group(2))
            salary_type = "upper_bound"
        elif above_match:
            salary_min = parse_numeric_token(above_match.group(2))
            salary_type = "lower_bound"
        elif single_match:
            salary_min = parse_numeric_token(single_match.group(1))
            salary_max = salary_min
            salary_type = "fixed"

    multiplier = detect_salary_multiplier(text, currency)
    salary_min_value = salary_min * multiplier if salary_min is not None else None
    salary_max_value = salary_max * multiplier if salary_max is not None else None

    usd_to_vnd = 25_000

    def to_vnd_month(x):
        if x is None:
            return None
        val = x * usd_to_vnd if currency == "usd" else x
        if period == "year":
            return val / 12
        if period == "day":
            return val * 22
        if period == "hour":
            return val * 8 * 22
        return val

    return {
        "salary_clean": text,
        "salary_min_value": salary_min_value,
        "salary_max_value": salary_max_value,
        "salary_currency": currency,
        "salary_period": period,
        "salary_type": salary_type,
        "salary_is_negotiable": negotiable,
        "salary_min_vnd_month": to_vnd_month(salary_min_value),
        "salary_max_vnd_month": to_vnd_month(salary_max_value),
    }

In [25]:
salary_examples = pd.Series([
    "12 - 20 triệu",
    "Thoả thuận",
    "Up to 30 triệu",
    "1000 USD",
    "300 triệu/năm",
    "Trên 20 triệu",
    "600 - 2,500 USD",
    "1,900 - 2,600 USD"
])

display(salary_examples.apply(parse_salary_range).apply(pd.Series))

,salary_clean,salary_min_value,salary_max_value,salary_currency,salary_period,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month
0,12 - 20 triệu,12000000.0,20000000.0,vnd,month,range,0,12000000.0,20000000.0
1,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
2,up to 30 triệu,NaN,30000000.0,vnd,month,upper_bound,0,NaN,30000000.0
3,1000 usd,1000.0,1000.0,usd,month,fixed,0,25000000.0,25000000.0
4,300 triệu/năm,300000000.0,300000000.0,vnd,year,fixed,0,25000000.0,25000000.0
5,trên 20 triệu,20000000.0,NaN,vnd,month,lower_bound,0,20000000.0,NaN
6,"600 - 2,500 usd",600.0,2500.0,usd,month,range,0,15000000.0,62500000.0
7,"1,900 - 2,600 usd",1900.0,2600.0,usd,month,range,0,47500000.0,65000000.0


In [26]:
salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)

salary_cols = [
    "salary_clean",
    "salary_min_value",
    "salary_max_value",
    "salary_currency",
    "salary_period",
    "salary_type",
    "salary_is_negotiable",
    "salary_min_vnd_month",
    "salary_max_vnd_month",
]

for col in salary_cols:
    df_clean[col] = salary_parsed.apply(lambda x: x[col])

display(
    df_clean[
        ["salary_raw"] + salary_cols
    ].head(10)
)

,salary_raw,salary_clean,salary_min_value,salary_max_value,salary_currency,salary_period,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month
0,12 - 20 triệu,12 - 20 triệu,12000000.0,20000000.0,vnd,month,range,0,12000000.0,20000000.0
1,20 - 30 triệu,20 - 30 triệu,20000000.0,30000000.0,vnd,month,range,0,20000000.0,30000000.0
2,30 - 35 triệu,30 - 35 triệu,30000000.0,35000000.0,vnd,month,range,0,30000000.0,35000000.0
3,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
4,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
5,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
6,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
7,6 - 9 triệu,6 - 9 triệu,6000000.0,9000000.0,vnd,month,range,0,6000000.0,9000000.0
8,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
9,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN


In [27]:
def clean_experience(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    return clean_text_strict(text)

def parse_experience_range(raw: str) -> dict:
    text = clean_experience(raw)
    if not text:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None,
            "experience_type": "unknown",
        }

    if any(k in text for k in ["không yêu cầu", "chưa có kinh nghiệm", "không cần kinh nghiệm"]):
        return {
            "experience_clean": text,
            "experience_min_years": 0.0,
            "experience_max_years": 0.0,
            "experience_type": "none_required",
        }

    month_match = re.search(r"(\d+(?:[.,]\d+)?)\s*(tháng|month)", text)
    if month_match:
        months = float(month_match.group(1).replace(",", "."))
        years = round(months / 12, 2)
        return {
            "experience_clean": text,
            "experience_min_years": years,
            "experience_max_years": years,
            "experience_type": "fixed",
        }

    if "dưới 1 năm" in text:
        return {
            "experience_clean": text,
            "experience_min_years": 0.0,
            "experience_max_years": 1.0,
            "experience_type": "range",
        }

    nums = [float(x.replace(",", ".")) for x in re.findall(r"\d+(?:[.,]\d+)?", text)]

    if len(nums) >= 2 and any(k in text for k in ["-", "đến", "tới"]):
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[1],
            "experience_type": "range",
        }

    if len(nums) >= 1:
        if any(k in text for k in ["từ", "trên", "ít nhất", ">=", "+"]):
            return {
                "experience_clean": text,
                "experience_min_years": nums[0],
                "experience_max_years": None,
                "experience_type": "minimum",
            }
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[0],
            "experience_type": "fixed",
        }

    return {
        "experience_clean": text,
        "experience_min_years": None,
        "experience_max_years": None,
        "experience_type": "unknown",
    }

In [28]:
exp_examples = pd.Series([
    "Không yêu cầu kinh nghiệm",
    "Dưới 1 năm",
    "Từ 2 năm",
    "2 - 4 năm",
    "Ít nhất 3 năm",
    "6 tháng",
    "1+ năm"
])

display(exp_examples.apply(parse_experience_range).apply(pd.Series))

,experience_clean,experience_min_years,experience_max_years,experience_type
0,không yêu cầu kinh nghiệm,0.0,0.0,none_required
1,dưới 1 năm,0.0,1.0,range
2,từ 2 năm,2.0,NaN,minimum
3,2 - 4 năm,2.0,4.0,range
4,ít nhất 3 năm,3.0,NaN,minimum
5,6 tháng,0.5,0.5,fixed
6,1+ năm,1.0,NaN,minimum


In [29]:
exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)

exp_cols = [
    "experience_clean",
    "experience_min_years",
    "experience_max_years",
    "experience_type"
]

for col in exp_cols:
    df_clean[col] = exp_parsed.apply(lambda x: x[col])

display(
    df_clean[
        ["experience_raw"] + exp_cols
    ].head(10)
)

,experience_raw,experience_clean,experience_min_years,experience_max_years,experience_type
0,2 năm,2 năm,2.0,2.0,fixed
1,2 năm,2 năm,2.0,2.0,fixed
2,2 năm,2 năm,2.0,2.0,fixed
3,3 năm,3 năm,3.0,3.0,fixed
4,2 năm,2 năm,2.0,2.0,fixed
5,3 năm,3 năm,3.0,3.0,fixed
6,3 năm,3 năm,3.0,3.0,fixed
7,Không yêu cầu,không yêu cầu,0.0,0.0,none_required
8,1 năm,1 năm,1.0,1.0,fixed
9,1 năm,1 năm,1.0,1.0,fixed


In [30]:
EDUCATION_MAP = {
    "trung học": "high_school",
    "thpt": "high_school",
    "cao đẳng": "college",
    "đại học": "bachelor",
    "cử nhân": "bachelor",
    "bachelor": "bachelor",
    "thạc sĩ": "master",
    "master": "master",
    "tiến sĩ": "phd",
    "phd": "phd",
}

def normalize_education_level(text: str) -> str:
    text = clean_text_strict(text)
    if not text:
        return "unknown"

    for key, value in EDUCATION_MAP.items():
        if key in text:
            return value
    return "unknown"

In [31]:
df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)

display(df_clean[["education_level_raw", "education_level_norm"]].head(10))

,education_level_raw,education_level_norm
0,Đại Học trở lên,bachelor
1,Đại Học trở lên,bachelor
2,Đại Học trở lên,bachelor
3,Đại Học trở lên,bachelor
4,Đại Học trở lên,bachelor
5,Đại Học trở lên,bachelor
6,Đại Học trở lên,bachelor
7,Đại Học trở lên,bachelor
8,Đại Học trở lên,bachelor
9,Đại Học trở lên,bachelor


In [32]:
EMPLOYMENT_TYPE_MAP = {
    "toàn thời gian": "full_time",
    "full time": "full_time",
    "full-time": "full_time",
    "bán thời gian": "part_time",
    "part time": "part_time",
    "part-time": "part_time",
    "thực tập": "internship",
    "intern": "internship",
    "internship": "internship",
    "freelance": "freelance",
    "cộng tác viên": "freelance",
    "hợp đồng": "contract",
    "contract": "contract",
    "temporary": "temporary",
}

def normalize_employment_type(text: str) -> str:
    text = clean_text_strict(text)
    if not text:
        return "unknown"

    for key, value in EMPLOYMENT_TYPE_MAP.items():
        if key in text:
            return value
    return "unknown"

In [33]:
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)

display(df_clean[["employment_type_raw", "employment_type_norm"]].head(10))

,employment_type_raw,employment_type_norm
0,Toàn thời gian,full_time
1,Toàn thời gian,full_time
2,Toàn thời gian,full_time
3,Toàn thời gian,full_time
4,Toàn thời gian,full_time
5,Toàn thời gian,full_time
6,Toàn thời gian,full_time
7,Toàn thời gian,full_time
8,Toàn thời gian,full_time
9,Toàn thời gian,full_time


In [34]:
JOB_LEVEL_RULES = [
    ("director", ["director", "giám đốc"]),
    ("manager", ["manager", "quản lý", "head", "trưởng phòng"]),
    ("lead", ["team lead", "trưởng nhóm", "leader", "lead", "principal", "staff"]),
    ("senior", ["senior", "sr", "cao cấp"]),
    ("middle", ["middle", "mid", "intermediate"]),
    ("junior", ["junior", "jr", "nhân viên", "chuyên viên"]),
    ("fresher", ["fresher", "entry level"]),
    ("intern", ["intern", "internship", "thực tập", "thực tập sinh"]),
]

def normalize_job_level(text: str, title_text: str = None) -> str:
    merged = " ".join([
        clean_text_strict(text),
        clean_text_strict(title_text)
    ]).strip()

    if not merged:
        return "unknown"

    for label, patterns in JOB_LEVEL_RULES:
        if any(re.search(rf"(?<!\w){re.escape(p)}(?!\w)", merged) for p in patterns):
            return label
    return "unknown"

In [35]:
df_clean["job_level_norm"] = df_clean.apply(
    lambda row: normalize_job_level(row.get("job_level_raw"), row.get("job_title_raw")),
    axis=1
)

display(
    df_clean[
        ["job_level_raw", "job_title_raw", "job_level_norm", "seniority_from_title"]
    ].head(10)
)

,job_level_raw,job_title_raw,job_level_norm,seniority_from_title
0,Nhân viên,Junior FP&A Analyst - Hồ Chí Minh,junior,junior
1,Nhân viên,Data Governance Specialist,junior,unknown
2,Nhân viên,Project Manager Dự Án AI HUB,manager,manager
3,Nhân viên,AI Engineer,junior,unknown
4,Nhân viên,AI Engineer,junior,unknown
5,Nhân viên,Data Analyst,junior,unknown
6,Nhân viên,Data Engineer,junior,unknown
7,Trưởng nhóm,Fresher Data Engineer,lead,junior
8,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics),lead,unknown
9,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,lead,unknown


In [36]:
metadata_preview_cols = [
    "job_title_raw", "job_title_canonical", "job_family", "seniority_from_title",
    "location_raw", "location_city", "location_district", "location_norm", "is_multi_location", "work_mode",
    "salary_raw", "salary_min_vnd_month", "salary_max_vnd_month", "salary_type", "salary_is_negotiable",
    "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
    "education_level_raw", "education_level_norm",
    "employment_type_raw", "employment_type_norm",
    "job_level_raw", "job_level_norm",
    "deadline_raw", "deadline_date", "days_to_deadline", "deadline_type"
]
metadata_preview_cols = [c for c in metadata_preview_cols if c in df_clean.columns]

display(df_clean[metadata_preview_cols].head(10))

,job_title_raw,job_title_canonical,job_family,seniority_from_title,location_raw,location_city,location_district,location_norm,is_multi_location,work_mode,salary_raw,salary_min_vnd_month,salary_max_vnd_month,salary_type,salary_is_negotiable,experience_raw,experience_min_years,experience_max_years,experience_type,education_level_raw,education_level_norm,employment_type_raw,employment_type_norm,job_level_raw,job_level_norm,deadline_raw,deadline_date,days_to_deadline,deadline_type
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst,other,junior,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,12 - 20 triệu,12000000.0,20000000.0,range,0,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,27/03/2026,2026-03-27,7,absolute_date
1,Data Governance Specialist,data governance specialist,data_governance_quality,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,20 - 30 triệu,20000000.0,30000000.0,range,0,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,Còn 17 ngày để ứng tuyển,2026-04-06,17,relative_days
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,product_project_ba,manager,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,30 - 35 triệu,30000000.0,35000000.0,range,0,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,manager,Còn 24 ngày để ứng tuyển,2026-04-13,24,relative_days
3,AI Engineer,ai engineer,data_science_ml,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,30/04/2026,2026-04-30,41,absolute_date
4,AI Engineer,ai engineer,data_science_ml,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Thoả thuận,NaN,NaN,unknown,1,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,11/05/2026,2026-05-11,52,absolute_date
5,Data Analyst,data analyst,data_analytics,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,08/04/2026,2026-04-08,19,absolute_date
6,Data Engineer,data engineer,data_engineering,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior,09/04/2026,2026-04-09,20,absolute_date
7,Fresher Data Engineer,fresher data engineer,data_engineering,junior,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,0,unknown,6 - 9 triệu,6000000.0,9000000.0,range,0,Không yêu cầu,0.0,0.0,none_required,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,09/04/2026,2026-04-09,20,absolute_date
8,Data Analyst Teamleader (Collection Analytics),data analyst teamleader collection analytics,data_analytics,unknown,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,Thoả thuận,NaN,NaN,unknown,1,1 năm,1.0,1.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,10/04/2026,2026-04-10,21,absolute_date
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,data analyst teamleader collection analytics,data_analytics,unknown,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,0,unknown,Thoả thuận,NaN,NaN,unknown,1,1 năm,1.0,1.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,10/04/2026,2026-04-10,21,absolute_date


In [37]:
normalize_check_cols = [
    "job_title_canonical",
    "location_norm",
    "salary_min_vnd_month",
    "experience_min_years",
    "education_level_norm",
    "employment_type_norm",
    "job_level_norm",
    "deadline_date"
]

report = {}
for col in normalize_check_cols:
    if col in df_clean.columns:
        if df_clean[col].dtype == "O":
            missing_ratio = df_clean[col].fillna("").astype(str).str.strip().eq("").mean()
        else:
            missing_ratio = df_clean[col].isna().mean()

        report[col] = {
            "missing_ratio": round(float(missing_ratio), 4),
            "nunique": int(df_clean[col].nunique(dropna=True))
        }

display(pd.DataFrame(report).T)

,missing_ratio,nunique
job_title_canonical,0.0000,259.0
location_norm,0.0000,9.0
salary_min_vnd_month,0.5662,31.0
experience_min_years,0.0000,6.0
education_level_norm,0.0000,4.0
employment_type_norm,0.0000,4.0
job_level_norm,0.0000,9.0
deadline_date,0.0000,39.0


# 4. Normalize tags, skill extraction và artifacts cho chatbot

In [38]:
def normalize_tags(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return []

    text = clean_text_light(text)
    parts = re.split(r"[,;/|\n]+", text)
    parts = [p.strip().lower() for p in parts if p and p.strip()]
    return deduplicate_list(parts)

df_clean["tags_list"] = df_clean["tags_raw"].apply(normalize_tags)

display(df_clean[["tags_raw", "tags_list"]].head(10))

,tags_raw,tags_list
0,Chuyên môn Data Analyst; Tài chính; Kế toán,"[chuyên môn data analyst, tài chính, kế toán]"
1,Chuyên môn Data Analyst,[chuyên môn data analyst]
2,Chuyên môn IT Project Manager,[chuyên môn it project manager]
3,Chuyên môn AI Engineer; IT - Phần mềm,"[chuyên môn ai engineer, it - phần mềm]"
4,Chuyên môn AI Engineer,[chuyên môn ai engineer]
5,Chuyên môn Data Analyst,[chuyên môn data analyst]
6,Chuyên môn Data Engineer,[chuyên môn data engineer]
7,Chuyên môn Data Engineer; IT - Phần mềm,"[chuyên môn data engineer, it - phần mềm]"
8,Chuyên môn Data Analyst; Tài chính; Ngân hàng; IT - Phần mềm,"[chuyên môn data analyst, tài chính, ngân hàng, it - phần mềm]"
9,Chuyên môn Data Analyst; Tài chính; Ngân hàng,"[chuyên môn data analyst, tài chính, ngân hàng]"


In [39]:
SKILL_TAXONOMY = {
    # programming / query
    "python": {"aliases": ["python", "python3"], "group": "programming"},
    "r": {"aliases": [" r ", "ngôn ngữ r"], "group": "programming"},
    "sql": {"aliases": ["sql", "mysql", "postgres", "postgresql", "oracle", "sql server"], "group": "database"},
    "java": {"aliases": ["java"], "group": "programming"},
    "scala": {"aliases": ["scala"], "group": "programming"},
    "c++": {"aliases": ["c++"], "group": "programming"},
    "c#": {"aliases": ["c#", "c sharp"], "group": "programming"},
    "javascript": {"aliases": ["javascript", "js"], "group": "programming"},
    "typescript": {"aliases": ["typescript", "ts"], "group": "programming"},

    # dataframe / DS libs
    "pandas": {"aliases": ["pandas"], "group": "data_science_stack"},
    "numpy": {"aliases": ["numpy"], "group": "data_science_stack"},
    "scikit-learn": {"aliases": ["scikit-learn", "sklearn"], "group": "data_science_stack"},
    "tensorflow": {"aliases": ["tensorflow"], "group": "ml_ai"},
    "pytorch": {"aliases": ["pytorch", "torch"], "group": "ml_ai"},

    # data engineering
    "etl": {"aliases": ["etl", "elt"], "group": "data_engineering"},
    "data warehouse": {"aliases": ["data warehouse", "dwh"], "group": "data_engineering"},
    "data lake": {"aliases": ["data lake", "lakehouse"], "group": "data_engineering"},
    "airflow": {"aliases": ["airflow"], "group": "data_engineering"},
    "dbt": {"aliases": ["dbt"], "group": "data_engineering"},
    "spark": {"aliases": ["spark", "apache spark", "pyspark"], "group": "data_engineering"},
    "hadoop": {"aliases": ["hadoop"], "group": "data_engineering"},
    "kafka": {"aliases": ["kafka", "apache kafka"], "group": "data_engineering"},
    "databricks": {"aliases": ["databricks"], "group": "data_engineering"},
    "bigquery": {"aliases": ["bigquery"], "group": "data_engineering"},
    "snowflake": {"aliases": ["snowflake"], "group": "data_engineering"},
    "redshift": {"aliases": ["redshift"], "group": "data_engineering"},

    # BI / reporting
    "power bi": {"aliases": ["power bi", "powerbi", "pbi"], "group": "bi_visualization"},
    "power query": {"aliases": ["power query"], "group": "bi_visualization"},
    "dax": {"aliases": ["dax"], "group": "bi_visualization"},
    "tableau": {"aliases": ["tableau"], "group": "bi_visualization"},
    "looker": {"aliases": ["looker"], "group": "bi_visualization"},
    "excel": {"aliases": ["excel"], "group": "bi_visualization"},
    "google data studio": {"aliases": ["google data studio", "data studio", "looker studio"], "group": "bi_visualization"},

    # analytics
    "statistics": {"aliases": ["statistics", "thống kê"], "group": "analytics"},
    "a/b testing": {"aliases": ["a/b testing", "ab testing", "a/b"], "group": "analytics"},
    "forecasting": {"aliases": ["forecast", "forecasting"], "group": "analytics"},

    # AI / ML
    "machine learning": {"aliases": ["machine learning", "machine-learning", "ml"], "group": "ml_ai"},
    "deep learning": {"aliases": ["deep learning", "deep-learning", "dl"], "group": "ml_ai"},
    "nlp": {"aliases": ["nlp", "natural language processing"], "group": "ml_ai"},
    "computer vision": {"aliases": ["computer vision", "cv"], "group": "ml_ai"},
    "ai": {"aliases": ["artificial intelligence", " ai "], "group": "ml_ai"},
    "gen ai": {"aliases": ["gen ai", "genai", "generative ai"], "group": "ml_ai"},
    "llm": {"aliases": ["llm", "large language model", "large language models"], "group": "ml_ai"},
    "rag": {"aliases": ["rag", "retrieval augmented generation"], "group": "ml_ai"},
    "langchain": {"aliases": ["langchain"], "group": "ml_ai"},

    # cloud / platform
    "aws": {"aliases": ["aws", "amazon web services"], "group": "cloud"},
    "gcp": {"aliases": ["gcp", "google cloud"], "group": "cloud"},
    "azure": {"aliases": ["azure"], "group": "cloud"},
    "docker": {"aliases": ["docker"], "group": "deployment"},
    "kubernetes": {"aliases": ["kubernetes", "k8s"], "group": "deployment"},
    "linux": {"aliases": ["linux"], "group": "deployment"},
    "git": {"aliases": ["git", "github", "gitlab"], "group": "deployment"},
    "ci/cd": {"aliases": ["ci/cd", "cicd", "continuous integration", "continuous delivery"], "group": "deployment"},
    "api": {"aliases": ["api", "rest api", "restful api"], "group": "deployment"},
    "fastapi": {"aliases": ["fastapi"], "group": "deployment"},
    "flask": {"aliases": ["flask"], "group": "deployment"},

    # governance / BA / PM
    "data governance": {"aliases": ["data governance"], "group": "governance"},
    "data quality": {"aliases": ["data quality"], "group": "governance"},
    "data lineage": {"aliases": ["data lineage"], "group": "governance"},
    "master data": {"aliases": ["master data"], "group": "governance"},
    "business analysis": {"aliases": ["business analysis", "business analyst", "phân tích nghiệp vụ"], "group": "product_project"},
    "project management": {"aliases": ["project management", "project manager", "quản lý dự án"], "group": "product_project"},
    "agile": {"aliases": ["agile"], "group": "product_project"},
    "scrum": {"aliases": ["scrum"], "group": "product_project"},
}

In [40]:
REQUIRED_HINTS = [
    "bắt buộc", "must have", "must", "yêu cầu", "required", "kinh nghiệm với",
    "thành thạo", "proficient", "strong knowledge", "am hiểu"
]
PREFERRED_HINTS = [
    "ưu tiên", "nice to have", "plus", "là lợi thế", "bonus", "ưu thế"
]

def alias_to_regex(alias: str):
    alias = alias.lower()
    if alias.strip() in {"r", "ml", "dl", "cv", "ai", "js", "ts"}:
        alias = alias.strip()
        return rf"(?<!\w){re.escape(alias)}(?!\w)"
    if alias.startswith(" ") or alias.endswith(" "):
        return rf"(?<!\w){re.escape(alias.strip())}(?!\w)"
    return rf"(?<!\w){re.escape(alias)}(?!\w)"

SKILL_PATTERNS = {
    skill_name: [re.compile(alias_to_regex(alias), flags=re.IGNORECASE) for alias in meta.get("aliases", [])]
    for skill_name, meta in SKILL_TAXONOMY.items()
}

def infer_skill_importance(segment: str, source_field: str) -> str:
    seg = clean_text_strict(segment)
    if any(h in seg for h in PREFERRED_HINTS):
        return "preferred"
    if source_field == "requirements":
        return "required"
    if any(h in seg for h in REQUIRED_HINTS):
        return "required"
    return "mentioned"

def extract_skill_records_from_text(text: str, source_field: str = "unknown"):
    text_clean = clean_text_preserve_structure(text)
    if not text_clean:
        return []

    segments = [seg.strip() for seg in re.split(r"\n+|[.;]", text_clean) if seg.strip()]
    records = []

    for seg in segments:
        for skill_name, patterns in SKILL_PATTERNS.items():
            if any(p.search(seg.lower()) for p in patterns):
                records.append({
                    "skill": skill_name,
                    "skill_group": SKILL_TAXONOMY[skill_name]["group"],
                    "source_field": source_field,
                    "importance": infer_skill_importance(seg, source_field),
                    "excerpt": seg[:220].strip(),
                })

    # dedup theo skill + source_field + importance
    dedup = []
    seen = set()
    for rec in records:
        key = (rec["skill"], rec["source_field"], rec["importance"])
        if key not in seen:
            seen.add(key)
            dedup.append(rec)
    return dedup

def skill_to_group(skill_name: str) -> str:
    return SKILL_TAXONOMY.get(skill_name, {}).get("group", "other")

In [41]:
df_clean["skill_records_from_tags"] = df_clean["tags_raw"].apply(lambda x: extract_skill_records_from_text(x, "tags"))
df_clean["skill_records_from_title"] = df_clean["job_title_clean_light"].apply(lambda x: extract_skill_records_from_text(x, "title"))
df_clean["skill_records_from_requirements"] = df_clean["requirements_clean_struct"].apply(lambda x: extract_skill_records_from_text(x, "requirements"))
df_clean["skill_records_from_description"] = df_clean["description_clean_struct"].apply(lambda x: extract_skill_records_from_text(x, "description"))

In [42]:
def merge_skill_records(*record_lists):
    merged = []
    seen = set()
    for lst in record_lists:
        if not isinstance(lst, list):
            continue
        for rec in lst:
            if not isinstance(rec, dict):
                continue
            key = (rec.get("skill"), rec.get("source_field"), rec.get("importance"))
            if key in seen:
                continue
            seen.add(key)
            merged.append(rec)
    return merged

def list_from_records(records, key, importance_filter=None):
    if not isinstance(records, list):
        return []
    values = []
    for rec in records:
        if importance_filter and rec.get("importance") not in importance_filter:
            continue
        values.append(rec.get(key))
    return deduplicate_list(values)

In [43]:
df_clean["skill_records"] = df_clean.apply(
    lambda row: merge_skill_records(
        row["skill_records_from_tags"],
        row["skill_records_from_title"],
        row["skill_records_from_requirements"],
        row["skill_records_from_description"]
    ),
    axis=1
)

df_clean["skills_extracted"] = df_clean["skill_records"].apply(lambda x: list_from_records(x, "skill"))
df_clean["skills_required"] = df_clean["skill_records"].apply(
    lambda x: list_from_records(x, "skill", importance_filter={"required"})
)
df_clean["skills_preferred"] = df_clean["skill_records"].apply(
    lambda x: list_from_records(x, "skill", importance_filter={"preferred"})
)
df_clean["skill_groups"] = df_clean["skill_records"].apply(lambda x: list_from_records(x, "skill_group"))

df_clean["skills_text"] = df_clean["skills_extracted"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df_clean["skills_required_text"] = df_clean["skills_required"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df_clean["skills_preferred_text"] = df_clean["skills_preferred"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df_clean["skill_groups_text"] = df_clean["skill_groups"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")

display(
    df_clean[
        [
            "job_title_raw",
            "tags_raw",
            "skills_extracted",
            "skills_required",
            "skills_preferred",
            "skill_groups"
        ]
    ].head(10)
)

,job_title_raw,tags_raw,skills_extracted,skills_required,skills_preferred,skill_groups
0,Junior FP&A Analyst - Hồ Chí Minh,Chuyên môn Data Analyst; Tài chính; Kế toán,[],[],[],[]
1,Data Governance Specialist,Chuyên môn Data Analyst,"[data governance, sql, etl, data warehouse, data quality]",[],"[data governance, sql, etl, data warehouse]","[governance, database, data_engineering]"
2,Project Manager Dự Án AI HUB,Chuyên môn IT Project Manager,"[project management, ai]","[project management, ai]",[ai],"[product_project, ml_ai]"
3,AI Engineer,Chuyên môn AI Engineer; IT - Phần mềm,"[ai, python, machine learning, llm]","[machine learning, ai, python, llm]","[python, machine learning, ai, llm]","[ml_ai, programming]"
4,AI Engineer,Chuyên môn AI Engineer,"[ai, llm, rag, langchain, api]","[ai, llm, rag, langchain]",[],"[ml_ai, deployment]"
5,Data Analyst,Chuyên môn Data Analyst,"[sql, etl, data warehouse, machine learning]","[sql, etl, data warehouse, machine learning]",[],"[database, data_engineering, ml_ai]"
6,Data Engineer,Chuyên môn Data Engineer,"[sql, etl, git]","[sql, etl, git]",[],"[database, data_engineering, deployment]"
7,Fresher Data Engineer,Chuyên môn Data Engineer; IT - Phần mềm,"[python, sql, scikit-learn, etl, spark, ai, power bi, aws]","[sql, etl, spark]","[python, sql, scikit-learn, etl, spark, ai]","[programming, database, data_science_stack, data_engineering, ml_ai, bi_visualization, cloud]"
8,Data Analyst Teamleader (Collection Analytics),Chuyên môn Data Analyst; Tài chính; Ngân hàng; IT - Phần mềm,"[python, sql, data warehouse]","[python, sql]",[],"[programming, database, data_engineering]"
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,Chuyên môn Data Analyst; Tài chính; Ngân hàng,"[python, sql, data warehouse]","[python, sql]",[],"[programming, database, data_engineering]"


In [44]:
skill_map_rows = []

for _, row in df_clean.iterrows():
    records = row.get("skill_records", [])
    if not isinstance(records, list):
        continue
    for rec in records:
        skill_map_rows.append({
            "job_url": row.get("job_url"),
            "job_title_canonical": row.get("job_title_canonical"),
            "job_family": row.get("job_family"),
            "skill": rec.get("skill"),
            "skill_group": rec.get("skill_group"),
            "source_field": rec.get("source_field"),
            "importance": rec.get("importance"),
            "excerpt": rec.get("excerpt"),
        })

job_skill_map_df = pd.DataFrame(skill_map_rows)

display(job_skill_map_df.head(10))

,job_url,job_title_canonical,job_family,skill,skill_group,source_field,importance,excerpt
0,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data governance,governance,title,mentioned,Data Governance Specialist
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data governance,governance,requirements,preferred,"- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,sql,database,requirements,preferred,"- Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế"
3,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,etl,data_engineering,requirements,preferred,"- Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế"
4,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data warehouse,data_engineering,requirements,preferred,"- Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế"
5,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data governance,governance,description,mentioned,"- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty"
6,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data quality,governance,description,mentioned,"- Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data security, metadata"
7,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data warehouse,data_engineering,description,mentioned,- Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI
8,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,project manager dự án ai hub,product_project_ba,project management,product_project,tags,mentioned,Chuyên môn IT Project Manager
9,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,project manager dự án ai hub,product_project_ba,ai,ml_ai,title,mentioned,Project Manager Dự Án AI HUB


In [45]:
df_clean["num_skills"] = df_clean["skills_extracted"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_clean["num_required_skills"] = df_clean["skills_required"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_clean["num_preferred_skills"] = df_clean["skills_preferred"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_clean["num_skill_groups"] = df_clean["skill_groups"].apply(lambda x: len(x) if isinstance(x, list) else 0)

display(
    df_clean[
        ["num_skills", "num_required_skills", "num_preferred_skills", "num_skill_groups"]
    ].describe()
)
print("Avg skills per job:", round(df_clean["num_skills"].mean(), 2))

,num_skills,num_required_skills,num_preferred_skills,num_skill_groups
count,325.000000,325.000000,325.000000,325.000000
mean,6.178462,5.058462,0.581538,3.061538
std,4.768612,4.358152,1.238639,1.980436
min,0.000000,0.000000,0.000000,0.000000
25%,2.000000,1.000000,0.000000,2.000000
50%,5.000000,4.000000,0.000000,3.000000
75%,10.000000,8.000000,1.000000,4.000000
max,22.000000,22.000000,9.000000,8.000000


Avg skills per job: 6.18


In [46]:
all_skills = df_clean["skills_extracted"].explode().dropna().tolist()
skill_counts = Counter(all_skills)
top_skills = pd.DataFrame(skill_counts.most_common(30), columns=["skill", "count"])
display(top_skills)

role_skill_stats_df = (
    job_skill_map_df
    .groupby(["job_family", "skill", "importance"], dropna=False)
    .size()
    .reset_index(name="job_count")
    .sort_values(["job_family", "job_count"], ascending=[True, False])
)

display(role_skill_stats_df.head(20))

,skill,count
0,ai,166
1,python,148
2,sql,143
3,machine learning,99
4,power bi,62
5,etl,55
6,api,55
7,statistics,53
8,spark,48
9,excel,48


,job_family,skill,importance,job_count
73,data_analytics,sql,required,45
58,data_analytics,power bi,required,28
79,data_analytics,tableau,required,21
63,data_analytics,python,required,19
76,data_analytics,statistics,required,16
36,data_analytics,excel,required,14
24,data_analytics,data warehouse,required,12
33,data_analytics,etl,required,12
56,data_analytics,power bi,mentioned,12
13,data_analytics,business analysis,mentioned,11


In [47]:
empty_skill_ratio = (df_clean["num_skills"] == 0).mean()
print(f"Tỷ lệ job không extract được skill: {round(empty_skill_ratio, 4)}")

Tỷ lệ job không extract được skill: 0.0954


In [48]:
def summarize_text(text: str, max_chars: int = 500) -> str:
    text = safe_str(text)
    if not text:
        return ""
    return text[:max_chars].strip()

def build_job_text_title(row):
    return safe_str(row.get("job_title_canonical") or row.get("job_title_display"))

def build_job_text_skills(row):
    parts = []
    if row.get("skills_required_text"):
        parts.append(f"kỹ năng bắt buộc: {row['skills_required_text']}")
    if row.get("skills_preferred_text"):
        parts.append(f"kỹ năng ưu tiên: {row['skills_preferred_text']}")
    if row.get("skills_text"):
        parts.append(f"tổng kỹ năng: {row['skills_text']}")
    return " | ".join([p for p in parts if p])

def build_job_text_requirements(row):
    req = summarize_text(row.get("requirements_clean_struct"), 900)
    exp = row.get("experience_min_years")
    edu = row.get("education_level_norm")
    pieces = []
    if req:
        pieces.append(req)
    if exp is not None and not pd.isna(exp):
        pieces.append(f"kinh nghiệm tối thiểu {exp} năm")
    if edu and edu != "unknown":
        pieces.append(f"trình độ {edu}")
    return "\n".join(pieces)

def build_job_text_description(row):
    return summarize_text(row.get("description_clean_struct"), 900)

def build_job_text_sparse(row):
    parts = [
        build_job_text_title(row),
        safe_str(row.get("skills_text")),
        safe_str(row.get("skill_groups_text")),
        summarize_text(row.get("requirements_clean_light"), 900),
        summarize_text(row.get("description_clean_light"), 700),
    ]
    return "\n".join([p for p in parts if p])

def build_job_text_dense(row):
    parts = []
    if row.get("job_title_display"):
        parts.append(f"tiêu đề: {row['job_title_display']}")
    if row.get("job_family") and row["job_family"] != "unknown":
        parts.append(f"nhóm nghề: {row['job_family']}")
    if row.get("skills_required_text"):
        parts.append(f"kỹ năng bắt buộc: {row['skills_required_text']}")
    if row.get("skills_preferred_text"):
        parts.append(f"kỹ năng ưu tiên: {row['skills_preferred_text']}")
    if row.get("requirements_clean_struct"):
        parts.append(f"yêu cầu:\n{summarize_text(row['requirements_clean_struct'], 1200)}")
    if row.get("description_clean_struct"):
        parts.append(f"mô tả công việc:\n{summarize_text(row['description_clean_struct'], 1000)}")
    return "\n\n".join([p for p in parts if p])

def build_job_text_chatbot(row):
    parts = []
    if row.get("job_title_display"):
        parts.append(f"Job title: {row['job_title_display']}")
    if row.get("job_family") and row["job_family"] != "unknown":
        parts.append(f"Job family: {row['job_family']}")
    if row.get("location_norm"):
        parts.append(f"Location: {row['location_norm']}")
    if row.get("work_mode") and row["work_mode"] != "unknown":
        parts.append(f"Work mode: {row['work_mode']}")
    if row.get("salary_min_vnd_month") or row.get("salary_max_vnd_month"):
        parts.append(
            f"Salary VND/month: min={row.get('salary_min_vnd_month')} | max={row.get('salary_max_vnd_month')}"
        )
    if row.get("requirements_clean_struct"):
        parts.append(f"Requirements:\n{summarize_text(row['requirements_clean_struct'], 1200)}")
    if row.get("description_clean_struct"):
        parts.append(f"Responsibilities:\n{summarize_text(row['description_clean_struct'], 1200)}")
    if row.get("benefits_clean_struct"):
        parts.append(f"Benefits:\n{summarize_text(row['benefits_clean_struct'], 800)}")
    if row.get("skills_required_text"):
        parts.append(f"Required skills: {row['skills_required_text']}")
    if row.get("skills_preferred_text"):
        parts.append(f"Preferred skills: {row['skills_preferred_text']}")
    return "\n\n".join(parts)

def build_job_chatbot_profile(row):
    return {
        "job_url": row.get("job_url"),
        "job_title": row.get("job_title_display"),
        "job_family": row.get("job_family"),
        "location": row.get("location_norm"),
        "work_mode": row.get("work_mode"),
        "experience_min_years": row.get("experience_min_years"),
        "education_level_norm": row.get("education_level_norm"),
        "skills_required": row.get("skills_required"),
        "skills_preferred": row.get("skills_preferred"),
    }

In [49]:
df_clean["job_text_title"] = df_clean.apply(build_job_text_title, axis=1)
df_clean["job_text_skills"] = df_clean.apply(build_job_text_skills, axis=1)
df_clean["job_text_requirements"] = df_clean.apply(build_job_text_requirements, axis=1)
df_clean["job_text_description"] = df_clean.apply(build_job_text_description, axis=1)

df_clean["job_text_sparse"] = df_clean.apply(build_job_text_sparse, axis=1)
df_clean["job_text_dense"] = df_clean.apply(build_job_text_dense, axis=1)
df_clean["job_text_chatbot"] = df_clean.apply(build_job_text_chatbot, axis=1)
df_clean["job_chatbot_profile"] = df_clean.apply(build_job_chatbot_profile, axis=1)

print("[INFO] Đã tạo xong job_text_sparse / dense / chatbot")

[INFO] Đã tạo xong job_text_sparse / dense / chatbot


In [50]:
display(
    df_clean[
        [
            "job_title_raw",
            "job_title_canonical",
            "skills_text",
            "skills_required_text",
            "job_text_title",
            "job_text_requirements",
            "job_text_sparse",
            "job_text_dense"
        ]
    ].head(3)
)

,job_title_raw,job_title_canonical,skills_text,skills_required_text,job_text_title,job_text_requirements,job_text_sparse,job_text_dense
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst,,,junior fp a analyst,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,junior fp a analyst\n- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree i...,tiêu đề: Junior FP A Analyst - Hồ Chí Minh\n\nnhóm nghề: other\n\nyêu cầu:\n- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in r...
1,Data Governance Specialist,data governance specialist,data governance; sql; etl; data warehouse; data quality,,data governance specialist,"- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","data governance specialist\ndata governance; sql; etl; data warehouse; data quality\ngovernance; database; data_engineering\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông...","tiêu đề: Data Governance Specialist\n\nnhóm nghề: data_governance_quality\n\nkỹ năng ưu tiên: data governance; sql; etl; data warehouse\n\nyêu cầu:\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ..."
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,project management; ai,project management; ai,project manager dự án ai hub,"Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...","project manager dự án ai hub\nproject management; ai\nproduct_project; ml_ai\nTối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng...",tiêu đề: Project Manager Dự Án AI HUB\n\nnhóm nghề: product_project_ba\n\nkỹ năng bắt buộc: project management; ai\n\nkỹ năng ưu tiên: ai\n\nyêu cầu:\nTối thiểu 2-3 năm kinh nghiệm ở vị trí Produc...


In [51]:
def split_long_text(text: str, max_chars: int = 700, overlap: int = 80):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    if not paragraphs:
        paragraphs = [text.strip()]

    chunks = []
    buffer = ""

    def flush_buffer():
        nonlocal buffer
        if buffer.strip():
            chunks.append(buffer.strip())
            buffer = ""

    for para in paragraphs:
        if len(para) <= max_chars:
            candidate = (buffer + "\n\n" + para).strip() if buffer else para
            if len(candidate) <= max_chars:
                buffer = candidate
            else:
                flush_buffer()
                buffer = para
        else:
            flush_buffer()
            start = 0
            while start < len(para):
                end = min(start + max_chars, len(para))
                piece = para[start:end].strip()
                if piece:
                    chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap)

    flush_buffer()
    return chunks

SECTION_PRIORITY = {
    "title": 1.5,
    "requirements": 1.3,
    "description": 1.0,
    "benefits": 0.6,
    "company": 0.5,
}

def build_job_section_records(row):
    section_specs = [
        ("title", row.get("job_text_title"), 220),
        ("requirements", row.get("requirements_clean_struct"), 700),
        ("description", row.get("description_clean_struct"), 700),
        ("benefits", row.get("benefits_clean_struct"), 600),
        ("company", row.get("company_description_clean_struct"), 600),
    ]

    rows = []
    chunk_order = 0
    for section_type, text, max_chars in section_specs:
        for chunk_text in split_long_text(text, max_chars=max_chars, overlap=80):
            chunk_order += 1
            rows.append({
                "job_url": row.get("job_url"),
                "job_title_display": row.get("job_title_display"),
                "job_title_canonical": row.get("job_title_canonical"),
                "job_family": row.get("job_family"),
                "section_type": section_type,
                "chunk_order": chunk_order,
                "chunk_text": chunk_text,
                "chunk_text_dense": chunk_text,
                "importance_weight": SECTION_PRIORITY.get(section_type, 1.0),
                "location_norm": row.get("location_norm"),
                "work_mode": row.get("work_mode"),
                "skills_required": row.get("skills_required"),
                "skills_preferred": row.get("skills_preferred"),
            })
    return rows

job_section_rows = []
for _, row in df_clean.iterrows():
    job_section_rows.extend(build_job_section_records(row))

job_sections_df = pd.DataFrame(job_section_rows)

display(job_sections_df.head(10))

,job_url,job_title_display,job_title_canonical,job_family,section_type,chunk_order,chunk_text,chunk_text_dense,importance_weight,location_norm,work_mode,skills_required,skills_preferred
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,title,1,junior fp a analyst,junior fp a analyst,1.5,hồ chí minh,unknown,[],[]
1,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,requirements,2,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,1.3,hồ chí minh,unknown,[],[]
2,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,description,3,"- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...","- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",1.0,hồ chí minh,unknown,[],[]
3,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,description,4,"ounting, Finance, and Other related functions to monitor and implement new policies, procedures, and internal control processes. - Make recommendations for the improvement & optimization of the ef...","ounting, Finance, and Other related functions to monitor and implement new policies, procedures, and internal control processes. - Make recommendations for the improvement & optimization of the ef...",1.0,hồ chí minh,unknown,[],[]
4,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,benefits,5,"- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...","- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",0.6,hồ chí minh,unknown,[],[]
5,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,company,6,"Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ...",0.5,hồ chí minh,unknown,[],[]
6,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,company,7,"17 năm phát triển và trưởng thành, Bee Logistics đã có sự phát triển vượt bậc với lực lượng nhân sự trên toàn hệ thống là hơn 900 người với các văn phòng tại Việt Nam (35 văn phòng), Cambodia, Mya...","17 năm phát triển và trưởng thành, Bee Logistics đã có sự phát triển vượt bậc với lực lượng nhân sự trên toàn hệ thốn

In [52]:
for col in ["job_text_title", "job_text_skills", "job_text_requirements", "job_text_description", "job_text_sparse", "job_text_dense", "job_text_chatbot"]:
    df_clean[f"{col}_len"] = df_clean[col].fillna("").str.len()

display(
    df_clean[
        [f"{c}_len" for c in ["job_text_title", "job_text_skills", "job_text_requirements", "job_text_description", "job_text_sparse", "job_text_dense", "job_text_chatbot"]]
    ].describe().T
)

print("=== VERY SHORT DENSE TEXT ===")
display(
    df_clean[df_clean["job_text_dense_len"] < 50][
        ["job_title_raw", "job_text_dense"]
    ].head(5)
)

print("=== VERY LONG CHATBOT TEXT ===")
display(
    df_clean[df_clean["job_text_chatbot_len"] > 2000][
        ["job_title_raw", "job_text_chatbot"]
    ].head(5)
)

print("=== SECTION COUNT BY TYPE ===")
display(job_sections_df["section_type"].value_counts(dropna=False))

,count,mean,std,min,25%,50%,75%,max
job_text_title_len,325.0,32.003077,19.371848,8.0,19.0,30.0,40.0,159.0
job_text_skills_len,325.0,134.144615,89.429022,0.0,69.0,121.0,190.0,419.0
job_text_requirements_len,325.0,720.273846,229.545435,166.0,538.0,755.0,948.0,950.0
job_text_description_len,325.0,711.772308,218.650188,149.0,535.0,821.0,900.0,900.0
job_text_sparse_len,325.0,1404.175385,353.093861,393.0,1194.0,1484.0,1702.0,1900.0
job_text_dense_len,325.0,1673.956923,515.991642,448.0,1295.0,1712.0,2089.0,2473.0
job_text_chatbot_len,325.0,2406.516923,674.331862,860.0,1921.0,2454.0,2928.0,3568.0


=== VERY SHORT DENSE TEXT ===


,job_title_raw,job_text_dense


=== VERY LONG CHATBOT TEXT ===


,job_title_raw,job_text_chatbot
0,Junior FP&A Analyst - Hồ Chí Minh,Job title: Junior FP A Analyst - Hồ Chí Minh\n\nJob family: other\n\nLocation: hồ chí minh\n\nSalary VND/month: min=12000000.0 | max=20000000.0\n\nRequirements:\n- At least 2 year’ experience in f...
1,Data Governance Specialist,"Job title: Data Governance Specialist\n\nJob family: data_governance_quality\n\nLocation: hà nội\n\nSalary VND/month: min=20000000.0 | max=30000000.0\n\nRequirements:\n- Đã tốt nghiệp ĐH trở lên, ..."
2,Project Manager Dự Án AI HUB,Job title: Project Manager Dự Án AI HUB\n\nJob family: product_project_ba\n\nLocation: hà nội\n\nSalary VND/month: min=30000000.0 | max=35000000.0\n\nRequirements:\nTối thiểu 2-3 năm kinh nghiệm ở...
3,AI Engineer,Job title: AI Engineer\n\nJob family: data_science_ml\n\nLocation: hà nội\n\nSalary VND/month: min=nan | max=nan\n\nRequirements:\nKinh nghiệm AI/ML ứng dụng thực tế Python và xử lý dữ liệu tốt Tư...
4,AI Engineer,"Job title: AI Engineer\n\nJob family: data_science_ml\n\nLocation: hà nội\n\nSalary VND/month: min=nan | max=nan\n\nRequirements:\nTốt nghiệp Đại học chuyên ngành CNTT, Khoa học máy tính, Điện tử ..."


=== SECTION COUNT BY TYPE ===


section_type
company         706
description     657
requirements    569
benefits        561
title           325
Name: count, dtype: int64

# 5. Dense embeddings cho matching (tuỳ chọn)

In [53]:
DENSE_MODEL_NAME = "vinai/phobert-base"
MULTILINGUAL_FALLBACK_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

if RUN_EMBEDDING:
    from transformers import AutoTokenizer, AutoModel
    import torch
    from tqdm import tqdm

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[INFO] Dense embedding device:", device)
    print("[INFO] Primary dense model   :", DENSE_MODEL_NAME)
else:
    device = None
    print("[INFO] RUN_EMBEDDING = False -> bỏ qua phần load model.")

[INFO] RUN_EMBEDDING = False -> bỏ qua phần load model.


In [54]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    denom = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / denom

def maybe_segment_vi_text(text: str) -> str:
    """
    PhoBERT hoạt động tốt hơn khi input đã word-segmented.
    Hàm này cố gắng dùng underthesea nếu có, nếu không thì fallback raw text.
    """
    text = safe_str(text)
    if not text:
        return ""

    try:
        from underthesea import word_tokenize
        return word_tokenize(text, format="text")
    except Exception:
        return text

In [55]:
if RUN_EMBEDDING:
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_NAME)
    model = AutoModel.from_pretrained(DENSE_MODEL_NAME)
    model.to(device)
    model.eval()

    def prepare_dense_input(text: str, encoder_route: str = "phobert") -> str:
        text = safe_str(text)
        if not text:
            return "[EMPTY]"
        if encoder_route == "phobert":
            return maybe_segment_vi_text(text)
        return text

    def encode_texts(texts, encoder_routes=None, batch_size=16, max_length=256):
        embeddings = []
        encoder_routes = encoder_routes or ["phobert"] * len(texts)

        prepared = [
            prepare_dense_input(text, route)
            for text, route in zip(texts, encoder_routes)
        ]

        for i in tqdm(range(0, len(prepared), batch_size)):
            batch = prepared[i:i + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)

            with torch.no_grad():
                model_output = model(input_ids=input_ids, attention_mask=attention_mask)

            batch_embeddings = mean_pooling(model_output, attention_mask)
            batch_embeddings = torch.nn.functional.normalize(batch_embeddings, p=2, dim=1)
            embeddings.append(batch_embeddings.cpu().numpy())

        return np.vstack(embeddings)
else:
    def encode_texts(texts, encoder_routes=None, batch_size=16, max_length=256):
        raise RuntimeError("RUN_EMBEDDING=False. Bật RUN_EMBEDDING ở cell 0 nếu muốn encode.")

In [56]:
job_dense_embeddings = None

if RUN_EMBEDDING:
    job_dense_embeddings = encode_texts(
        df_clean["job_text_dense"].fillna("").tolist(),
        encoder_routes=df_clean["dense_encoder_route"].fillna("phobert").tolist(),
        batch_size=16,
        max_length=256,
    )
    df_clean["has_dense_embedding"] = 1
    print("job_dense_embeddings:", job_dense_embeddings.shape)
else:
    df_clean["has_dense_embedding"] = 0
    print("[INFO] Skip job-level embedding.")

[INFO] Skip job-level embedding.


In [57]:
section_dense_embeddings = None

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and len(job_sections_df) > 0:
    section_dense_embeddings = encode_texts(
        job_sections_df["chunk_text_dense"].fillna("").tolist(),
        encoder_routes=["phobert"] * len(job_sections_df),
        batch_size=16,
        max_length=256,
    )
    job_sections_df["has_dense_embedding"] = 1
    print("section_dense_embeddings:", section_dense_embeddings.shape)
else:
    job_sections_df["has_dense_embedding"] = 0
    print("[INFO] Skip section-level embedding.")

[INFO] Skip section-level embedding.


In [58]:
DOWNSTREAM_FIELD_GUIDE = {
    "tfidf_input": "job_text_sparse",
    "dense_input": "job_text_dense",
    "chatbot_job_summary": "job_text_chatbot",
    "chatbot_chunk_table": "job_sections_df",
    "skill_table": "job_skill_map_df",
    "role_skill_stats": "role_skill_stats_df",
}

display(pd.DataFrame(DOWNSTREAM_FIELD_GUIDE.items(), columns=["use_case", "field_or_table"]))

,use_case,field_or_table
0,tfidf_input,job_text_sparse
1,dense_input,job_text_dense
2,chatbot_job_summary,job_text_chatbot
3,chatbot_chunk_table,job_sections_df
4,skill_table,job_skill_map_df
5,role_skill_stats,role_skill_stats_df


In [59]:
matching_cols = [
    "job_url",
    "source_field_name",
    "job_title_raw",
    "job_title_display",
    "job_title_canonical",
    "job_family",
    "seniority_from_title",
    "location_norm",
    "location_city",
    "location_district",
    "is_multi_location",
    "work_mode",
    "salary_min_vnd_month",
    "salary_max_vnd_month",
    "salary_type",
    "salary_is_negotiable",
    "experience_min_years",
    "experience_max_years",
    "experience_type",
    "education_level_norm",
    "employment_type_norm",
    "job_level_norm",
    "deadline_date",
    "days_to_deadline",
    "skills_extracted",
    "skills_required",
    "skills_preferred",
    "skill_groups",
    "job_text_title",
    "job_text_skills",
    "job_text_requirements",
    "job_text_description",
    "job_text_sparse",
    "job_text_dense",
    "dense_encoder_route",
]

df_matching_ready = df_clean[[c for c in matching_cols if c in df_clean.columns]].copy()
display(df_matching_ready.head(3))

,job_url,source_field_name,job_title_raw,job_title_display,job_title_canonical,job_family,seniority_from_title,location_norm,location_city,location_district,is_multi_location,work_mode,salary_min_vnd_month,salary_max_vnd_month,salary_type,salary_is_negotiable,experience_min_years,experience_max_years,experience_type,education_level_norm,employment_type_norm,job_level_norm,deadline_date,days_to_deadline,skills_extracted,skills_required,skills_preferred,skill_groups,job_text_title,job_text_skills,job_text_requirements,job_text_description,job_text_sparse,job_text_dense,dense_encoder_route
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,junior,hồ chí minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,0,unknown,12000000.0,20000000.0,range,0,2.0,2.0,fixed,bachelor,full_time,junior,2026-03-27,7,[],[],[],[],junior fp a analyst,,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,"- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",junior fp a analyst\n- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree i...,tiêu đề: Junior FP A Analyst - Hồ Chí Minh\n\nnhóm nghề: other\n\nyêu cầu:\n- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in r...,phobert
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,Data Governance Specialist,Data Governance Specialist,data governance specialist,data_governance_quality,unknown,hà nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,0,unknown,20000000.0,30000000.0,range,0,2.0,2.0,fixed,bachelor,full_time,junior,2026-04-06,17,"[data governance, sql, etl, data warehouse, data quality]",[],"[data governance, sql, etl, data warehouse]","[governance, database, data_engineering]",data governance specialist,kỹ năng ưu tiên: data governance; sql; etl; data warehouse | tổng kỹ năng: data governance; sql; etl; data warehouse; data quality,"- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","data governance specialist\ndata governance; sql; etl; data warehouse; data quality\ngovernance; database; data_engineering\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông...","tiêu đề: Data Governance Specialist\n\nnhóm nghề: data_governance_quality\n\nkỹ năng ưu tiên: data governance; sql; etl; data warehouse\n\nyêu cầu:\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ...",phobert
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,AI Engineer,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,product_project_ba,manager,hà nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,0,unknown,30000000.0,35000000.0,range,0,2.0,2.0,fixed,bachelor,full_time,manager,2026-04-13,24,"[project management, ai]","[project management, ai]",[ai],"[product_project, ml_ai]",project manager dự án ai hub,kỹ năng bắt buộc: project management; ai | kỹ năng ưu tiên: ai | tổng kỹ năng: project management; ai,"Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng th

In [60]:
chatbot_cols = [
    "job_url",
    "job_title_display",
    "job_title_canonical",
    "job_family",
    "location_norm",
    "work_mode",
    "deadline_date",
    "skills_extracted",
    "skills_required",
    "skills_preferred",
    "skill_groups",
    "requirements_clean_struct",
    "description_clean_struct",
    "benefits_clean_struct",
    "job_text_chatbot",
    "job_chatbot_profile",
]

df_chatbot_ready = df_clean[[c for c in chatbot_cols if c in df_clean.columns]].copy()
display(df_chatbot_ready.head(3))

,job_url,job_title_display,job_title_canonical,job_family,location_norm,work_mode,deadline_date,skills_extracted,skills_required,skills_preferred,skill_groups,requirements_clean_struct,description_clean_struct,benefits_clean_struct,job_text_chatbot,job_chatbot_profile
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,hồ chí minh,unknown,2026-03-27,[],[],[],[],- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,"- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...","- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Job title: Junior FP A Analyst - Hồ Chí Minh\n\nJob family: other\n\nLocation: hồ chí minh\n\nSalary VND/month: min=12000000.0 | max=20000000.0\n\nRequirements:\n- At least 2 year’ experience in f...,"{'job_url': 'https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html', 'job_title': 'Junior FP A Analyst - Hồ Chí Minh', 'job_family': 'other', 'l..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Governance Specialist,data governance specialist,data_governance_quality,hà nội,unknown,2026-04-06,"[data governance, sql, etl, data warehouse, data quality]",[],"[data governance, sql, etl, data warehouse]","[governance, database, data_engineering]","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...","Job title: Data Governance Specialist\n\nJob family: data_governance_quality\n\nLocation: hà nội\n\nSalary VND/month: min=20000000.0 | max=30000000.0\n\nRequirements:\n- Đã tốt nghiệp ĐH trở lên, ...","{'job_url': 'https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html', 'job_title': 'Data Governance Specialist', 'job_family': 'data_governance_quality', 'location': ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,Project Manager Dự Án AI HUB,project manager dự án ai hub,product_project_ba,hà nội,unknown,2026-04-13,"[project management, ai]","[project management, ai]",[ai],"[product_project, ml_ai]","Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...",1. Lập kế hoạch & Xây dựng chiến lược AI (20%) Xây dựng roadmap triển khai AI Hub nhằm nâng cao năng suất các bộ phận trong công ty. Phân tích quy trình vận hành hiện tại của các phòng ban để xác ...,"Tinh thần: - Làm việc trong môi trường start-up có tốc độ tăng trưởng và phát triển nhanh. - Được chia sẻ cơ hội học tập chất lượng cao đến trẻ em toàn Việt Nam - Môi trường làm việc: Trẻ trung, n...",Job title: Project Manager Dự Án AI HUB\n\nJob family: product_project_ba\n\nLocation: hà nội\n\nSalary VND/month: min=30000000.0 | max=35000000.0\n\nRequirements:\nTối thiểu 2-3 năm kinh nghiệm ở...,"{'job_url': 'https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an

In [61]:
job_sections_ready = job_sections_df.copy()
display(job_sections_ready.head(3))

,job_url,job_title_display,job_title_canonical,job_family,section_type,chunk_order,chunk_text,chunk_text_dense,importance_weight,location_norm,work_mode,skills_required,skills_preferred,has_dense_embedding
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,title,1,junior fp a analyst,junior fp a analyst,1.5,hồ chí minh,unknown,[],[],0
1,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,requirements,2,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,1.3,hồ chí minh,unknown,[],[],0
2,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst,other,description,3,"- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...","- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",1.0,hồ chí minh,unknown,[],[],0


In [62]:
job_skill_map_ready = job_skill_map_df.copy()
display(job_skill_map_ready.head(3))

,job_url,job_title_canonical,job_family,skill,skill_group,source_field,importance,excerpt
0,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data governance,governance,title,mentioned,Data Governance Specialist
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,data governance,governance,requirements,preferred,"- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,data governance specialist,data_governance_quality,sql,database,requirements,preferred,"- Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế"


In [63]:
role_skill_stats_ready = role_skill_stats_df.copy()
display(role_skill_stats_ready.head(10))

,job_family,skill,importance,job_count
73,data_analytics,sql,required,45
58,data_analytics,power bi,required,28
79,data_analytics,tableau,required,21
63,data_analytics,python,required,19
76,data_analytics,statistics,required,16
36,data_analytics,excel,required,14
24,data_analytics,data warehouse,required,12
33,data_analytics,etl,required,12
56,data_analytics,power bi,mentioned,12
13,data_analytics,business analysis,mentioned,11


In [64]:
matching_path = save_table(df_matching_ready, ARTIFACT_DIR / "jobs_matching_ready_v2")
chatbot_path = save_table(df_chatbot_ready, ARTIFACT_DIR / "jobs_chatbot_ready_v2")
sections_path = save_table(job_sections_ready, ARTIFACT_DIR / "jobs_chatbot_sections_v2")
skill_map_path = save_table(job_skill_map_ready, ARTIFACT_DIR / "job_skill_map_v2")
role_skill_stats_path = save_table(role_skill_stats_ready, ARTIFACT_DIR / "role_skill_stats_v2")

embedding_paths = {}
if RUN_EMBEDDING and job_dense_embeddings is not None:
    job_dense_path = ARTIFACT_DIR / "job_dense_embeddings_v2.npy"
    np.save(job_dense_path, job_dense_embeddings)
    embedding_paths["job_dense_embeddings"] = str(job_dense_path)

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and section_dense_embeddings is not None:
    section_dense_path = ARTIFACT_DIR / "job_section_dense_embeddings_v2.npy"
    np.save(section_dense_path, section_dense_embeddings)
    embedding_paths["section_dense_embeddings"] = str(section_dense_path)

print("[INFO] Saved main artifacts.")

[INFO] Saved main artifacts.


In [65]:
artifact_summary = pd.DataFrame({
    "artifact_name": [
        "jobs_matching_ready_v2",
        "jobs_chatbot_ready_v2",
        "jobs_chatbot_sections_v2",
        "job_skill_map_v2",
        "role_skill_stats_v2",
    ],
    "path": [
        str(matching_path),
        str(chatbot_path),
        str(sections_path),
        str(skill_map_path),
        str(role_skill_stats_path),
    ]
})

display(artifact_summary)

,artifact_name,path
0,jobs_matching_ready_v2,D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_matching_ready_v2.parquet
1,jobs_chatbot_ready_v2,D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_chatbot_ready_v2.parquet
2,jobs_chatbot_sections_v2,D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_chatbot_sections_v2.parquet
3,job_skill_map_v2,D:\TTTN\Project\outputs_preprocessing_v2\artifacts\job_skill_map_v2.parquet
4,role_skill_stats_v2,D:\TTTN\Project\outputs_preprocessing_v2\artifacts\role_skill_stats_v2.parquet


In [66]:
manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_date": RUN_DATE,
    "raw_input_path": str(RAW_INPUT_PATH),
    "artifacts": {
        "matching_ready": str(matching_path),
        "chatbot_ready": str(chatbot_path),
        "chatbot_sections": str(sections_path),
        "job_skill_map": str(skill_map_path),
        "role_skill_stats": str(role_skill_stats_path),
    },
    "embedding_paths": embedding_paths,
    "downstream_field_guide": DOWNSTREAM_FIELD_GUIDE,
}

manifest_path = ARTIFACT_DIR / "preprocessing_manifest_v2.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("[INFO] Saved manifest:", manifest_path)
display(pd.Series(manifest))

[INFO] Saved manifest: D:\TTTN\Project\outputs_preprocessing_v2\artifacts\preprocessing_manifest_v2.json


notebook_version                                                                                                                                                                                                 preprocessing_v2
run_date                                                                                                                                                                                                      2026-03-20 17:49:44
raw_input_path                                                                                                                                        D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-16_20-57-23.xlsx
artifacts                 {'matching_ready': 'D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_matching_ready_v2.parquet', 'chatbot_ready': 'D:\TTTN\Project\outputs_preprocessing_v2\artifacts\jobs_chatbot_ready_v2.p...
embedding_paths                                                                                 

### Ghi chú sử dụng
- `jobs_matching_ready_v2`: dùng cho nhánh TF-IDF / cosine / filter metadata.
- `jobs_chatbot_ready_v2`: dùng cho chatbot ở mức job summary.
- `jobs_chatbot_sections_v2`: dùng cho RAG / retrieval theo section.
- `job_skill_map_v2`: dùng cho skill gap analysis và explainability.
- `role_skill_stats_v2`: dùng cho thống kê kỹ năng phổ biến theo nhóm job.

Khuyến nghị tiếp theo:
1. Parse CV về cùng schema với job.
2. Dùng `job_text_sparse` cho TF-IDF/BM25.
3. Dùng `job_text_dense` + `job_sections_ready` cho dense retrieval / rerank.
4. Chỉ bật embedding khi artifact text đã ổn.